In [6]:
import subprocess
import sys

packages = ["projectz", "zestio"]
for package in packages:

    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [7]:
from model_engine.assets.utils import load_asset, get_default_factory_configs
from model_engine.model_builder.asset_parser import asset_parser
from model_engine.client_predictor.client_predictor import ClientPredictor
from model_engine.assets.power.app_asset import schema
from model_engine.model_builder import ModelBuilder
from model_engine.io.utils import global_load_data, global_archive_data, global_path_exists
from model_engine.power.post_sale.base.utils import normalize_io
from model_engine.power.post_sale.base import ClientHandler, ModelConstraints, PowerModelingBase, S3ClientHandler
from model_engine.power.post_sale import ClientModelBuilder
from model_engine.io.loaders import DictHelper
from model_engine.io.s3_schema_manager import S3SchemaManager
import s3fs
import pandas as pd
from zestio import DataArchiver
import copy
from projectz import ZInfo
from zestio import handler
from zestio.handler import FileSystemHandler, S3Handler
import json
import os
from zestio import handler
from zaml.common.utils.io import load_state
from zestio import handler

In [8]:
def connect_client(z_info, client_name):
    z_info.connect_client(name = client_name)
    return z_info
def connect_datasets(z_info, dataset_version):
    datasets = z_info.context.web_handler.get(f"clients/{z_info.client.id}/datasets")
    dataset_num = dataset_version-1
    dataset_model_type = datasets[dataset_num]['model_type']
    dataset_data_source = datasets[dataset_num]['data_source']
    dataset_product = datasets[dataset_num]['product']
    dataset_version = datasets[dataset_num]['version']
    z_info.connect_dataset(model_type = dataset_model_type,
                                     product = dataset_product, 
                                     version = dataset_version)
    return z_info
def connect_model(z_info, model_name, product):
    z_info.connect_model_iteration(name = model_name, product = product,  model_type=['underwriting'])
    return z_info
def connect_to_z_info(dataset_version, product, model_name, client_name):
    z_info = ZInfo()
    z_info =connect_client(z_info, client_name)
    z_info = connect_datasets(z_info, dataset_version)
    z_info = connect_model(z_info, model_name, product)
    return z_info

# ADD YOUR NAME HERE SO YOU CAN SAVE IN A NEW LOCATION

In [ ]:
#your_name = 'gazin'

your_name = 'None' # replace here

In [18]:
if your_name is None:
    raise ValueError("Please set your_name variable to your name before running the notebook.")

In [19]:
dataset_version = 1
product = ['autoloan']
model_name = 'model1_AppData_LTVAutoloan'
client_name = 'californiacu'

z_info = connect_to_z_info(dataset_version, product, model_name, client_name)

INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected to client: californiacu


INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected to dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected to model_iteration: autoloan/model1_AppData_LTVAutoloan


In [20]:
model_artifacts_dir = z_info.model_iteration._dir_paths['modeling_model_artifacts_dir']

# Note our configuration_api is saved [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L607) 
# This configuration_api is what we build in our 1_run_model1.ipynb and pass into the PowerModelBuilder. It is like a map that tells the PowerModelBuilder how to build the model
# The goal of this notebook is to trace how all the information passes down and builds our final ensemble model

In [128]:
final_model_path =  's3://power-model-artifacts-staging/POST_SALE_DELIVERY/SCHEMA_VERSION=v1/CLIENT=californiacu/BUREAU=equifax/PRODUCT=autoloan/PRODUCT_VERSION=v1/MODEL=model1_AppData_LTVAutoloan/'
client_only_model_path = 's3://power-model-artifacts-staging/CLIENT_ONLY_MODEL/SCHEMA_VERSION=v1/CLIENT=californiacu/BUREAU=equifax/PRODUCT=autoloan/PRODUCT_VERSION=v1/MODEL=model1_AppData_LTVAutoloan/'

In [129]:
os.path.join(model_artifacts_dir, 'power_model_config.json')

'/d/shared/silver_projects_v2/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/model_artifacts/autoloan/model1_AppData_LTVAutoloan/power_model_config.json'

In [130]:
configuration_api = json.load(open(os.path.join(model_artifacts_dir, 'power_model_config.json')))

In [131]:
configuration_api.keys()

dict_keys(['client_data', 'client_model', 'national_model', 'national_samples_evaluation', 'evaluate_client_model_only', 'skip_argument_validation', 'schema', 'api_version', 'fe_version'])

In [132]:
configuration_api['client_data']

{'storage_location': 's3',
 'clients': ['californiacu_autoloan'],
 's3clients': [{'client': 'californiacu',
   'product': 'autoloan',
   'version': 'v2',
   'bureau': 'equifax',
   'pull_name': '20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu',
   'pull_date': '2026-02-01',
   'me_version': 'v2.1.1'}],
 'asset': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_type': 'app',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'numerics': ['AmountRequested', 'TotalVehicleValue'],
   'categoricals': [],
   'feature_definitions': {'AmountRequested': 'to be removed',
    'TotalVehicleValue': 'to be removed'},
   'placeholders': {'encoding_type': 'one-hot',
    'placeholders': {'AmountRequested': {'NaN': {'flag': False,
       'val': '9999999999'},
      '0': {'flag': False, 'val': '9999999999'},
      '0.0': {'flag': False, 'val': '9999999999'},
      '0.00': {'flag': False, 'val': '9999999999'}},
     'TotalVehicleValue'

In [133]:
configuration_api['client_model']

{'existing_model_path': None,
 'model_config': {'bivariate_fe_instructions': [{'feature_1': 'client_data_AmountRequested',
    'feature_2': 'client_data_TotalVehicleValue',
    'operation': 'divide',
    'new_feature': 'biv_loan_to_value'}],
  'monotonic_constraints_list': [{'feature': 'biv_loan_to_value',
    'monotonic_constraint': 'positive'}],
  'exclusion_list': [{'feature': 'client_data_AmountRequested',
    'reason': 'used indirectly'},
   {'feature': 'client_data_TotalVehicleValue', 'reason': 'used indirectly'},
   {'feature': 'TotalVehicleValue', 'reason': 'Used Indirectly'},
   {'feature': 'MonthlyIncomeBaseSalary', 'reason': 'Not Used'},
   {'feature': 'AmountRequested', 'reason': 'Used Indirectly'},
   {'feature': 'MonthlyIncome', 'reason': 'Not Used'},
   {'feature': 'VehicleValue', 'reason': 'Not Used'},
   {'feature': 'LoanTerm', 'reason': 'Not Used'},
   {'feature': 'LTV', 'reason': 'Not Used'},
   {'feature': 'DTI', 'reason': 'Not Used'},
   {'feature': 'PTI', 'reason'

In [134]:
configuration_api['national_model']

{'existing_model_path': None,
 'model_config': {'bureau': 'equifax',
  'product': 'auto',
  'model_type': 'mega_model_member_data_fe2'},
 'model_settings': {'product': 'autoloan',
  'region': 'west',
  'KOB': 'creditunion'}}

In [135]:
configuration_api['national_samples_evaluation']

In [136]:
configuration_api['evaluate_client_model_only']

False

In [137]:
configuration_api['skip_argument_validation']

False

In [138]:
configuration_api['schema']

'v3'

In [139]:
configuration_api['api_version']

'v1'

In [140]:
configuration_api['fe_version']

2

# We are going to trace the backend of the code when we call [pmb.run()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L589-L608) starting with [run_combination](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L608)

```python
pmb = PowerModelBuilder(
    configuration_api, 
    final_model_output_path, 
    client_model_output_path
)
pmb.run()
```

# Note, that [get_fold_valid](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L172) will return True 

## In our case, we will not have an existing client model path and we will not be evaluating on national samples. The first check is whether we have a validation set.

Since `configuration_api['client_model']['model_config']['fold_valid']` is `True`, this means we will:

- Train the client model first with the validation held out (Pass 1 - supporting model)
- Find the best number of estimators from early stopping
- Then fold the validation into training and retrain with that exact number of estimators (Pass 2)

In [141]:
configuration_api['client_model']['model_config']['fold_valid']

True

# We therefore first call [make_client_model_build_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L550). This will start preparing our configuration for the ClientModelBuilder object for the model with the valid set held out. At this point, we are still just training the Client Model: the first iteration whose main goal is to find the optimal number of estimators

```python
def make_client_model_build_config(self):
    dh = DictHelper(self.configuration_api)
    cmb_config = {
        'clients': dh.get_value(['client_data', 'clients']),
        's3clients': dh.get_value(['client_data', 's3clients']),
        'storage_location': dh.get_value(['client_data', 'storage_location']),
        'client_data_asset': dh.get_value(['client_data', 'asset']),
        'model_config': dh.get_value(['client_model', 'model_config']) or {},
        'bureau': self.get_bureau(),
        'schema': dh.get_value(['schema']),
        'model_settings': dh.get_value(['national_model', 'model_settings']),
        'fe_version': self.fe_version
    }
    cmb_config = self._adapt_model_config(cmb_config)
    cmb_config = self._include_model_setting(cmb_config)
    return cmb_config
```

In [142]:

dh = DictHelper(configuration_api)
cmb_config = {
            'clients': dh.get_value(['client_data', 'clients']),
            's3clients': dh.get_value(['client_data', 's3clients']),
            'storage_location': dh.get_value(['client_data', 'storage_location']),
            'client_data_asset': dh.get_value(['client_data', 'asset']),
            'model_config': dh.get_value(['client_model', 'model_config']) or {},
            'bureau': dh.get_value(['national_model', 'model_config', 'bureau'  ]), ## edited so we grab bureau from the configuration_Api
            'schema': dh.get_value(['schema']),
            'model_settings': dh.get_value(['national_model', 'model_settings']),
            'fe_version': dh.get_value(['fe_version']) ## edited so we grab version
        }


## We then call [adapt_model_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L210). Note, for the bureau features, we want our national model and client model to allign, this is what the adapt_model_config is doing
```python
def _adapt_model_config(self, cmb_config):
        dh = DictHelper(cmb_config)
        dh.insert_key_value(['model_config'], {'feature_definition_list': dh.get_value(['model_config', 'feature_definition_list'], []) + self._get_model_optional_model_settings('feature_definition_list')})
        dh.insert_key_value(['model_config'], {'key_factor_mapping_list': dh.get_value(['model_config', 'key_factor_mapping_list'], []) + self._get_model_optional_model_settings('key_factor_mapping_list')})
        dh.insert_key_value(['model_config'], {'base_features': self.get_base_features()})
        return dh.get_data()
        
 def _get_model_optional_model_settings(self, setting_name):
        if not self.is_using_mega_model():
            return []
        mega_model_config_options = global_load_data(os.path.join(self.get_national_model_path(), 'megamodel_config_options.json'))
        return mega_model_config_options.get(setting_name) or []
 def get_base_features(self):
        '''
        This method extracts the base features from the national model.
        '''
        national_model_path = self.get_national_model_path()
        base_feature_path = os.path.join(national_model_path, 'keep_features.json')
        if global_path_exists(base_feature_path):
            return base_feature_path
        else:
            raise PowerModelBuilderError(f'{national_model_path} does not contain keep_features.json file.')  
```

This updates the client predictor config to have the base features, key factor mapping, and feature definition list of the national model.

Note, our national model path is fond with [get_national_model_path](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L134) which is based off the model type we feed 

In [143]:
configuration_api['national_model']['model_config']

{'bureau': 'equifax',
 'product': 'auto',
 'model_type': 'mega_model_member_data_fe2'}

## Note, that our model_type is mega_model_member_data_fe2. This is because this is a credit union so we can include member-features in our national model. These are features from tradelines that were at credit unions.

In [144]:
national_model_paths = load_asset('power/post_sale_path.json')

### See below that the Asset for the National Mega Model does not have member data whereas the asset for the mega model member data does.

In [145]:
json.load(open(os.path.join(national_model_paths['mega_model_fe2'], 'asset.json')))['data'].keys()

dict_keys(['app', 'trade', 'inq', 'collec', 'bnkr', 'config', 'target'])

In [146]:
json.load(open(os.path.join(national_model_paths['mega_model_member_data_fe2'], 'asset.json')))['data'].keys()

dict_keys(['app', 'trade', 'inq', 'collec', 'bnkr', 'member', 'target', 'config'])

In [147]:
correct_national_model_path = national_model_paths['mega_model_member_data_fe2']

## This is the path for the national model. We can see it has tons of files in this path related to that national model

In [148]:
correct_national_model_path

'/d/shared/national_model_warehouse/national_models/fe2/VERSION=211/BUREAU=mega/PRODUCT=mega/FEATURE=v2_member/ITERATION=1'

In [149]:
os.listdir(correct_national_model_path)

['splitter.obj',
 'feature_definition_original.parquet',
 'overall_zest_scores',
 'asset.json',
 'value_based_key_factor_mapping.json',
 'feature_definition.parquet',
 'fe_data',
 'fb_analysis_results.parquet',
 'unfold_data_description.json',
 'lookalike_sample',
 'artifact_manifest.json',
 'performance.parquet',
 'pipeline_experian.obj',
 'model.obj',
 '.ipynb_checkpoints',
 'national_predictor_config.json',
 'fbm.obj',
 'test_data_summary.json',
 'app',
 'pipeline_equifax.obj',
 'spark_artifacts',
 'versions.json',
 'data_description.json',
 'power_sampler_config.json',
 'score_recalibration_mapping.json',
 'key_factors_mapping.json',
 'feature_importance.parquet',
 'overall_scores',
 'megamodel_config_options.json',
 'national_model_config.json',
 'keep_features.json',
 'model_strategy.json',
 'dta_result.parquet',
 'client_predictor_config.json',
 'target',
 'calibration_object.obj',
 'spark_build_configuration.json',
 'pipeline_transunion.obj',
 'pipeline.obj']

In [150]:
## this is just loading the feature definition list and the key factors from our national model
feature_definition_list = global_load_data(os.path.join(correct_national_model_path, 'megamodel_config_options.json'))['feature_definition_list']
key_factor_mapping_list = global_load_data(os.path.join(correct_national_model_path, 'megamodel_config_options.json'))['key_factor_mapping_list']

## get_base features is just grabbing the keep_features from our model

base_features = os.path.join(correct_national_model_path, 'keep_features.json')

In [151]:
helper = DictHelper(copy.deepcopy(cmb_config)) ## calling deep copy so we can see what changed
helper.insert_key_value(['model_config'], {'feature_definition_list': helper.get_value(['model_config', 'feature_definition_list'], []) + feature_definition_list}) 
helper.insert_key_value(['model_config'], {'key_factor_mapping_list': helper.get_value(['model_config', 'key_factor_mapping_list'], []) + key_factor_mapping_list})
helper.insert_key_value(['model_config'], {'base_features': base_features})

{'clients': ['californiacu_autoloan'],
 's3clients': [{'client': 'californiacu',
   'product': 'autoloan',
   'version': 'v2',
   'bureau': 'equifax',
   'pull_name': '20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu',
   'pull_date': '2026-02-01',
   'me_version': 'v2.1.1'}],
 'storage_location': 's3',
 'client_data_asset': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_type': 'app',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'numerics': ['AmountRequested', 'TotalVehicleValue'],
   'categoricals': [],
   'feature_definitions': {'AmountRequested': 'to be removed',
    'TotalVehicleValue': 'to be removed'},
   'placeholders': {'encoding_type': 'one-hot',
    'placeholders': {'AmountRequested': {'NaN': {'flag': False,
       'val': '9999999999'},
      '0': {'flag': False, 'val': '9999999999'},
      '0.0': {'flag': False, 'val': '9999999999'},
      '0.00': {'flag': False, 'val': '9999999999'}},
     'TotalV

In [152]:
cmb_config_after_dapt = helper.get_data()

In [153]:
cmb_config['model_config'].get('base_features')

In [154]:
cmb_config['model_config'].get('key_factor_mapping_list')

[{'feature': 'biv_loan_to_value', 'key_factor': 'Deal characteristic'}]

## We can see we have the key factors from the national model added

In [155]:
cmb_config_after_dapt['model_config'].get('key_factor_mapping_list')

[{'feature': 'biv_loan_to_value', 'key_factor': 'Deal characteristic'},
 {'feature': 'config_KOB_flg_creditunion', 'key_factor': None},
 {'feature': 'config_KOB_flg_regionalbank', 'key_factor': None},
 {'feature': 'config_product_flg_autoloan', 'key_factor': None},
 {'feature': 'config_product_flg_creditcard', 'key_factor': None},
 {'feature': 'config_product_flg_personalloan', 'key_factor': None},
 {'feature': 'config_product_flg_homeequity', 'key_factor': None},
 {'feature': 'config_product_flg_businessloan', 'key_factor': None},
 {'feature': 'config_region_flg_west', 'key_factor': None},
 {'feature': 'config_region_flg_northeast', 'key_factor': None},
 {'feature': 'config_region_flg_south', 'key_factor': None},
 {'feature': 'config_region_flg_midwest', 'key_factor': None}]

In [156]:
cmb_config_after_dapt['model_config']['base_features']

'/d/shared/national_model_warehouse/national_models/fe2/VERSION=211/BUREAU=mega/PRODUCT=mega/FEATURE=v2_member/ITERATION=1/keep_features.json'

In [157]:
len(cmb_config_after_dapt['model_config']['feature_definition_list']), len(cmb_config['model_config']['key_factor_mapping_list'])

(12, 1)

In [158]:
cmb_config_after_dapt['model_config']['key_factor_mapping_list']

[{'feature': 'biv_loan_to_value', 'key_factor': 'Deal characteristic'},
 {'feature': 'config_KOB_flg_creditunion', 'key_factor': None},
 {'feature': 'config_KOB_flg_regionalbank', 'key_factor': None},
 {'feature': 'config_product_flg_autoloan', 'key_factor': None},
 {'feature': 'config_product_flg_creditcard', 'key_factor': None},
 {'feature': 'config_product_flg_personalloan', 'key_factor': None},
 {'feature': 'config_product_flg_homeequity', 'key_factor': None},
 {'feature': 'config_product_flg_businessloan', 'key_factor': None},
 {'feature': 'config_region_flg_west', 'key_factor': None},
 {'feature': 'config_region_flg_northeast', 'key_factor': None},
 {'feature': 'config_region_flg_south', 'key_factor': None},
 {'feature': 'config_region_flg_midwest', 'key_factor': None}]

## We then include the model settings [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L212)

* These model setting help control how the mega model config flag features will be set during inference.
* For example, since we know this is a credit-union from the west whose product is autoloan, we will want to set the flag for config_KOB_flg_creditunion to be True, config_product_flg_autoloan to be True, and config_region_flg_west to be True
```python
def _include_model_setting(self, cmb_config):
        model_settings = DictHelper(self.configuration_api).get_value(['national_model', 'model_settings'])
        if model_settings:
            config_asset = load_asset('megamodel/config.json')
            config_asset['info']['constant_categorical_feature'] = model_settings
            config_asset['info']['placeholders']['placeholders'] = self._make_placeholders_mega_model()
            cmb_config['mega_model_config_asset'] = config_asset
        return cmb_config
```

with [_make_placeholders_mega_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L467-L477)

```python
def _make_placeholders_mega_model(self):
        mega_model_settings = global_load_data(os.path.join(self.get_national_model_path(), 'megamodel_config_options.json'))
        placeholders = {}
        for key, values in mega_model_settings['config'].items():
            placeholders[key] = {
                value['val']: {
                    'flag': value['encode'],
                    'val': value['val']
                } for value in values if value['encode']
            }
        return placeholders
```




In [159]:
os.listdir(os.path.join(model_artifacts_dir, 'fe_data', 'split=test', 'data=client'))


['data_0.parquet']

In [160]:
client_test_data = pd.read_parquet(os.path.join(model_artifacts_dir, 'fe_data', 'split=test', 'data=client', 'data_0.parquet'))

In [161]:
flg_columns = [feature for feature in client_test_data.columns if 'flg' in feature]

## We can see that in our final FE Data that config_KOB_flg_creditunion=1, config_region_flg_west=1, config_product_flg_autoloan=1 and the rest are 0!

In [162]:
client_test_data[flg_columns].drop_duplicates()

,config_KOB_flg_creditunion,config_KOB_flg_regionalbank,config_product_flg_autoloan,config_product_flg_businessloan,config_product_flg_creditcard,config_product_flg_homeequity,config_product_flg_personalloan,config_region_flg_midwest,config_region_flg_northeast,config_region_flg_south,config_region_flg_west
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [163]:
flg_columns = [feature for feature in client_test_data.columns if 'flg' in feature]

In [164]:
model_settings = DictHelper(configuration_api).get_value(['national_model', 'model_settings'])


In [165]:
model_settings

{'product': 'autoloan', 'region': 'west', 'KOB': 'creditunion'}

In [166]:
config_asset = load_asset('megamodel/config.json')

In [167]:
config_asset

{'feature_engineering': 'one_to_one',
 'validator': 'base',
 'table_name': 'config',
 'table_type': 'one_to_one',
 'info': {'key': 'ZEST_KEY',
  'app_date': 'appDate',
  'categoricals': {},
  'constant_categorical_feature': 'DICTTOBEUPDATED',
  'placeholders': {'encoding_type': 'one-hot',
   'drop_original_features': True,
   'placeholders': 'DICTTOBEUPDATED'},
  'feature_definitions': {'product': 'Product',
   'region': 'Region',
   'KOB': 'KOB'}}}

### This constant categorical features part right here is telling the model to do that

In [168]:
config_asset['info']['constant_categorical_feature'] = model_settings

## The placeholders code is giving us instructions on how we will create the flg columns: we want them to be created the same way as our mega-model

In [169]:
mega_model_settings = global_load_data(os.path.join(correct_national_model_path, 'megamodel_config_options.json'))
placeholders = {}
for key, values in mega_model_settings['config'].items():
    placeholders[key] = {
        value['val']: {
            'flag': value['encode'],
            'val': value['val']
        } for value in values if value['encode']
    }

In [170]:
mega_model_settings['config']

{'KOB': [{'encode': True, 'val': 'creditunion'},
  {'encode': True, 'val': 'regionalbank'}],
 'product': [{'encode': True, 'val': 'autoloan'},
  {'encode': True, 'val': 'creditcard'},
  {'encode': True, 'val': 'personalloan'},
  {'encode': True, 'val': 'homeequity'},
  {'encode': True, 'val': 'businessloan'},
  {'encode': False, 'val': 'allproducts'}],
 'region': [{'encode': True, 'val': 'west'},
  {'encode': True, 'val': 'northeast'},
  {'encode': True, 'val': 'south'},
  {'encode': True, 'val': 'midwest'},
  {'encode': False, 'val': 'national'}]}

* We can see below that there are 2 KOB flag columns which matches the config above
* There are 5 product flg columns which matches the config above
* And 4 region columns which matches the config above
* So the placeholders tell us what flg columns to create and the constant_categorical_feature takes our model_settings as directions for which one to turn on

In [171]:
[feature for feature in flg_columns if 'region' in feature]

['config_KOB_flg_regionalbank',
 'config_region_flg_midwest',
 'config_region_flg_northeast',
 'config_region_flg_south',
 'config_region_flg_west']

In [172]:
[feature for feature in flg_columns if 'product' in feature]

['config_product_flg_autoloan',
 'config_product_flg_businessloan',
 'config_product_flg_creditcard',
 'config_product_flg_homeequity',
 'config_product_flg_personalloan']

In [173]:
[feature for feature in flg_columns if 'KOB' in feature]

['config_KOB_flg_creditunion', 'config_KOB_flg_regionalbank']

In [174]:
placeholders

{'KOB': {'creditunion': {'flag': True, 'val': 'creditunion'},
  'regionalbank': {'flag': True, 'val': 'regionalbank'}},
 'product': {'autoloan': {'flag': True, 'val': 'autoloan'},
  'creditcard': {'flag': True, 'val': 'creditcard'},
  'personalloan': {'flag': True, 'val': 'personalloan'},
  'homeequity': {'flag': True, 'val': 'homeequity'},
  'businessloan': {'flag': True, 'val': 'businessloan'}},
 'region': {'west': {'flag': True, 'val': 'west'},
  'northeast': {'flag': True, 'val': 'northeast'},
  'south': {'flag': True, 'val': 'south'},
  'midwest': {'flag': True, 'val': 'midwest'}}}

In [175]:
config_asset['info']['placeholders']['placeholders'] = placeholders

In [176]:
cmb_config_after_dapt['mega_model_config_asset'] = config_asset

# We now have our client model builder config: our instructions for creating our first Client Model Builder Object have been created. We now need to build the object and run it

# We now call [build_client_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L551) with is_supporting_model True to build our model with the fold left out.

```python
def build_client_model(self, client_model_build_input, is_supporting_model):
        '''
        This method create the client-only model based on given client dataset.
        '''
        if is_supporting_model:
            client_model_output_path = os.path.join(self.client_model_output_path, 'supporting_model')
        else:
            client_model_output_path = self.client_model_output_path
        cmb = ClientModelBuilder(client_model_build_input, client_model_output_path)
        client_model_path, pipeline = cmb.run()
        return client_model_path, pipeline
```

In [177]:
cmb = ClientModelBuilder(cmb_config_after_dapt, client_only_model_path)

In [178]:
cmb.__dict__.keys()

dict_keys(['config', 'fe_version', 'power_asset', 'possible_bureaus', 'schema', 'storage_location', 'output_system', 'model_config', 'data_split', 'lookalike', 'bureau', 'bureau_asset', 'lookalike_obj', 'artifacts_path', 'model_constraint_obj', 'enforce_artifact_validation', 'client_handler', 'target_spec', 's3clients'])

## Our config attribute matches the config we just created

In [179]:
cmb.config == cmb_config_after_dapt

True

In [180]:
cmb.artifacts_path

's3://power-model-artifacts-staging/CLIENT_ONLY_MODEL/SCHEMA_VERSION=v1/CLIENT=californiacu/BUREAU=equifax/PRODUCT=autoloan/PRODUCT_VERSION=v1/MODEL=model1_AppData_LTVAutoloan/'

## We still have the valid in this data split

In [181]:
cmb.data_split ### has the valid data split

{'train': {'start_date': '2021-07-01', 'end_date': '2022-11-05'},
 'valid': {'start_date': '2022-11-05', 'end_date': '2023-04-21'},
 'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}}

In [182]:
cmb.data_split== cmb_config_after_dapt['model_config']['data_split']

True

In [183]:
cmb.power_asset ## This is based on national model data

{'data_path': {'file_system': '/d/shared/silver_projects_v2/project_power/shared_data/processed/national_data/fe2/',
  's3': 'national_data/'},
 'data_split': {'train': {'start_date': '2016-01-01',
   'end_date': '2021-04-01'},
  'valid': {'start_date': '2021-04-01', 'end_date': '2021-07-01'},
  'test': {'start_date': '2021-07-01', 'end_date': '2022-01-01'}}}

In [184]:
cmb.config

{'clients': ['californiacu_autoloan'],
 's3clients': [{'client': 'californiacu',
   'product': 'autoloan',
   'version': 'v2',
   'bureau': 'equifax',
   'pull_name': '20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu',
   'pull_date': '2026-02-01',
   'me_version': 'v2.1.1'}],
 'storage_location': 's3',
 'client_data_asset': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_type': 'app',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'numerics': ['AmountRequested', 'TotalVehicleValue'],
   'categoricals': [],
   'feature_definitions': {'AmountRequested': 'to be removed',
    'TotalVehicleValue': 'to be removed'},
   'placeholders': {'encoding_type': 'one-hot',
    'placeholders': {'AmountRequested': {'NaN': {'flag': False,
       'val': '9999999999'},
      '0': {'flag': False, 'val': '9999999999'},
      '0.0': {'flag': False, 'val': '9999999999'},
      '0.00': {'flag': False, 'val': '9999999999'}},
     'TotalV

## We now call run which after archiving files creates our [data section](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L330-L332). This will be another map that tells us where to grab the data to train our client model, how we should prepare the data (the asset), along with IO parameters (input/output) which wil control how the data is loaded with data loader.

```python
data_section = self._create_empty_data_section()
data_section = self._inject_config(data_section)
data_section = self._inject_client_data(data_section, self.storage_location)
```

```python
def _create_empty_data_section(self):
    result = {}
    asset = self.bureau_asset['asset']
    base_feature_space = self._get_base_feature_space(self.config['model_config']['base_features'])
    # Example bureau_asset: ./model_engine/assets/power/experian.json
    # Get bureau tables whose features are used in the modeling
    bureau_tables = list(base_feature_space)
    # Apart from bureau tables, two additional tables are also used in modeling
    base_tables = ['app', 'target']
    for name, entity in asset.items():
        # Only bureau tables whose features are used in modeling shall be included in the asset
        # Meanwhile, base_tables should always be included
        if name not in base_tables and name not in bureau_tables:
            continue
        result[name] = {}
        result[name]['asset'] = entity
        result[name]['data'] = []
        result[name]['io_params'] = {"drop_duplicates": True}
        if name == 'app' and 'keep_keys' in self.config['client_data_asset']['io_params']:
            result[name]['io_params']['keep_keys'] = self.config['client_data_asset']['io_params']['keep_keys']
    return result
```



In [185]:
cmb.bureau_asset.keys()

dict_keys(['loan_type', 'pull_name', 'partition', 'date_range', 'asset'])

In [186]:
cmb.data_split

{'train': {'start_date': '2021-07-01', 'end_date': '2022-11-05'},
 'valid': {'start_date': '2022-11-05', 'end_date': '2023-04-21'},
 'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}}

In [187]:
cmb.bureau_asset['date_range']

{'m24': {'start': '2016-01-01', 'end': '2021-10-01'}}

In [188]:
cmb.bureau_asset['asset']

{'trade': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
  'AggregationEngine': 'aggregation/fe2/trade.json'},
 'inq': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/inquiry.json',
  'AggregationEngine': 'aggregation/fe2/inquiry.json'},
 'collec': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/collections.json',
  'AggregationEngine': 'aggregation/fe2/collections.json'},
 'bnkr': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/bankruptcy.json',
  'AggregationEngine': 'aggregation/fe2/bankruptcy.json'},
 'member': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/member.json',
  'AggregationEngine': 'aggregation/fe2/member.json'},
 'auth': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/authorized.json',
  'AggregationEngine': 'aggregation/fe2/authorized.json'},
 'non_ddp_inq': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/non_dedupe_inquiry.json',
  'AggregationEngine': 'aggregation/fe2/non_dedupe_inq

In [189]:
base_feature_path = cmb.config['model_config']['base_features']

In [190]:
base_feature_path

'/d/shared/national_model_warehouse/national_models/fe2/VERSION=211/BUREAU=mega/PRODUCT=mega/FEATURE=v2_member/ITERATION=1/keep_features.json'

# We can see that [_create_empty_data_section()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L187) creates the skeleton of this data map
* It reads `keep_features.json` from the national model to figure out which bureau tables are actually needed, and grabs the assets that tell us how to load and prepare the data
* When this step completes, we won't have the locations of the data we'll load - just a map saying these are the bureau tables we want and these are the instructions (the assets) for how to process them
* Note these assets include both Normalization and Aggregation specs. At the same time, our data paths will point to already-processed (feature-engineered) data

## Why do we still need these normalization and aggregation assets when our data is already fully processed?
* Because we want our final pipeline object to contain the full instructions for taking data from preprocessed → feature-engineered. That pipeline is what runs in production, where the data has only been preprocessed - not yet feature-engineered
* The feature engine transformers handle this with a pass-through: they check whether their output columns already exist in the input. If they do (training), the transformer does nothing. If they don't (production), it runs the full FE steps. So the same pipeline works in both contexts

In [191]:
data_section = {}
asset = cmb.bureau_asset['asset']
base_feature_space = global_load_data(base_feature_path) ## replaced with actual data
# Example bureau_asset: ./model_engine/assets/power/experian.json
# Get bureau tables whose features are used in the modeling
bureau_tables = list(base_feature_space)
# Apart from bureau tables, two additional tables are also used in modeling
base_tables = ['app', 'target']
for name, entity in asset.items():
    # Only bureau tables whose features are used in modeling shall be included in the asset
    # Meanwhile, base_tables should always be included
    if name not in base_tables and name not in bureau_tables:
        continue
    data_section[name] = {}
    data_section[name]['asset'] = entity
    data_section[name]['data'] = []
    data_section[name]['io_params'] = {"drop_duplicates": True}
    if name == 'app' and 'keep_keys' in cmb.config['client_data_asset']['io_params']:
        data_section[name]['io_params']['keep_keys'] = cmb.config['client_data_asset']['io_params']['keep_keys']

In [192]:
data_section.keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target'])

## At this point, our data section only contains which bureau tables we need and the instructions (assets) for how to process them - there are no data paths pointing to actual data yet

In [193]:
import pprint

for key, value in data_section.items():
    print(f"\n=== {key} ===")
    pprint.pprint(value)


=== trade ===
{'asset': {'AggregationEngine': 'aggregation/fe2/trade.json',
           'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}

=== inq ===
{'asset': {'AggregationEngine': 'aggregation/fe2/inquiry.json',
           'InputNormalizer': 'equifax/cms_6/fe2/inquiry.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}

=== collec ===
{'asset': {'AggregationEngine': 'aggregation/fe2/collections.json',
           'InputNormalizer': 'equifax/cms_6/fe2/collections.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}

=== bnkr ===
{'asset': {'AggregationEngine': 'aggregation/fe2/bankruptcy.json',
           'InputNormalizer': 'equifax/cms_6/fe2/bankruptcy.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}

=== member ===
{'asset': {'AggregationEngine': 'aggregation/fe2/member.json',
         

# [_inject_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L209) passes the mega model config instructions down into our data section so the pipeline knows how to prepare the config flag features
```python
def _inject_config(self, data_section):
    if self.config.get('model_settings'):
        data_section['config'] = {}
        data_section['config']['data'] = []
        data_section['config']['io_params'] = {"drop_duplicates": True}        
        data_section['config']['asset'] = self.config['mega_model_config_asset']
    return data_section
```

* Up to this point, our data section only has bureau tables (trade, inq, collec, etc.) plus app and target. These are tables backed by real data on S3
* `_inject_config` adds a virtual table called `config` that has no actual data rows (`data: []`) - it exists purely to carry the mega model config asset we built earlier with `_include_model_setting()`
* That asset contains the `constant_categorical_feature` (which model_settings values to stamp onto every row) and the `placeholders` (which flag columns to one-hot encode and how)
* When the pipeline later processes this table through `OneToOneEngine`, `AddCountFeature` will stamp the constant values and `ProcessPlaceholders` will create the 11 binary flag columns
* So where `_create_empty_data_section` told the pipeline "here are the bureau tables and how to process them," `_inject_config` tells it "here is also a config table, and here are the instructions for generating the mega model segment features"

In [194]:
cmb.config.get('model_settings')

{'product': 'autoloan', 'region': 'west', 'KOB': 'creditunion'}

In [195]:
data_section['config'] = {}
data_section['config']['data'] = []
data_section['config']['io_params'] = {"drop_duplicates": True}        
data_section['config']['asset'] = cmb.config['mega_model_config_asset'] ## this is the loaded mega model asset with the placeholders and the constant categorical features updated with model settings

In [196]:
cmb.storage_location

's3'

# [_inject_client_data](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L217) wires up the actual S3 data paths into our data section - up to now we had the instructions but no data locations
```python
def _inject_client_data(self, data_section: dict, storage_location: Literal['dshared', 's3'] = 'dshared'):
        # Initialize the client data handler where all necessary client data related info is stored in its attributes
        if storage_location == 'dshared':
            self.client_handler = ClientHandler(
                data_section=data_section, 
                clients=self.config['clients'], 
                client_data_asset=self.config['client_data_asset'],
                target_table = self.target_spec['target_table']
            )
        else:
            self.client_handler = S3ClientHandler(
                data_section=data_section, 
                s3clients=self.config['s3clients'], 
                client_data_asset=self.config['client_data_asset'],
                target_table = self.target_spec['target_table']
            )
        # Inject the client data path into data section
        client_data_section = copy.deepcopy(self.client_handler.data_section)
        for table, entity in self.client_handler.data_section.items():
            client_data_section[table]['data'] = self.client_handler.client_data.get(table, [])
        # Normalize the data section to include feature space
        client_data_section = normalize_io(client_data_section, self.config['model_config'], keep_features=list(schema[self.schema].keys()), ignore_sources=['config'])
        return client_data_section
```
* It creates an [S3ClientHandler](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L12) which does the heavy lifting during `__init__`
* The core of this is [make_client_data](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L67) - it takes the s3client parameters we passed in (client, product, version, bureau, pull_name, pull_date, me_version) and uses `S3SchemaManager` to resolve the actual S3 path for each table (trade, inq, collec, etc.). It also checks that required tables actually exist on S3 and raises an error if they don't
* `make_client_data` also calls [add_client_data_to_data_section](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L97) which adds the `client_data` table (our app features like AmountRequested, TotalVehicleValue) into the data section and sets `is_client` as a categorical on the app table
* `_update_data_section_data` is a no-op for us - it only does something when data paths are JSON file references (the `#path.json` pattern used in dshared mode). Since our data values are empty lists at this point, the `isinstance(data, str)` check fails and nothing happens
* After the handler is built, `_inject_client_data` deep copies the handler's data section, overwrites each table's `data` with the resolved S3 paths from `client_data`, and then calls `normalize_io` to inject `keep_features` into each table's `io_params`


```python
def __init__(self, data_section:dict, s3clients: list[dict], client_data_asset: dict=None, target_table:str=None):
        self.data_section = copy.deepcopy(data_section)
        self.client_data_asset = client_data_asset
        self.target_table = target_table or 'target'
        self.s3clients = s3clients
        self.validate_arguments()
        self.client_data = self.make_client_data(s3clients, self.target_table)
        self._update_data_section_data()
```


## Initialization of S3ClientHandler: make_client_data

* We first load [data_table_info](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L37) - this is a registry that maps every possible data table (trade, inq, collec, target, etc.) to its metadata: the S3 database name, the pipeline input node it maps to, and whether it's optional or required.

In [197]:
data_table_info = load_asset('power/data_table_info.json')

In [198]:
data_table_info.keys()

dict_keys(['app', 'client_data', 'target', 'trade', 'inquiry', 'collection', 'bankruptcy', 'member', 'authorized', 'non_dedupe_inquiry', 'lexis_nexis', 'trended_noplaceholders_fe', 'corev2_5707_noplaceholders_fe', 'corev2_5708_noplaceholders_fe', 'config'])

In [199]:
for key in data_table_info.keys():
    print(f"\n=== {key} ===")
    pprint.pprint(data_table_info[key])


=== app ===
{'asset_file': None,
 'bureau_specific': False,
 'client_db': 'app.parquet',
 'data_source': 'client',
 'optional': False,
 'pipeline_fe_node': 'app FE',
 'pipeline_input_node': 'app',
 's3_db': 'analysis_data'}

=== client_data ===
{'asset_file': None,
 'bureau_specific': False,
 'client_db': 'client_data.parquet',
 'data_source': 'client',
 'optional': False,
 'pipeline_fe_node': 'client_data FE',
 'pipeline_input_node': 'client_data',
 's3_db': 'analysis_data'}

=== target ===
{'asset_file': None,
 'bureau_specific': False,
 'client_db': 'target.parquet',
 'data_source': 'client',
 'optional': False,
 'pipeline_fe_node': 'target FE',
 'pipeline_input_node': 'target',
 's3_db': 'target'}

=== trade ===
{'asset_file': 'trade.json',
 'bureau_specific': True,
 'client_db': 'trade.parquet',
 'data_source': 'bureau',
 'normalized_asset': 'trade_fe.json',
 'optional': False,
 'pipeline_fe_node': 'trade FE',
 'pipeline_input_node': 'trade',
 's3_db': 'trade'}

=== inquiry ===
{

### We initialize instance attributes in __init__ [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L48-L52)

### We then create a client_data attribute [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L53) by calling make_client_data(s3_clients, self.target_table)
```python
def make_client_data(self, s3clients: list[dict], target_table: str) -> dict:
    """
    Generate a dict where key is the data table name and the value is the path to different tables of client data: trade, inquiry, etc
    Be aware that the 'client_data' table is NOT included here.
    """
    data = {info['pipeline_input_node']: [] for _, info in self.data_table_info.items()}
    for client in s3clients:
        for table_name, table_info in self.data_table_info.items():
            if table_name != 'target':
                table = table_info['s3_db']
            else:
                table = target_table
            spec = {
                'TABLE': table,
                'VERSION': client['version'], 
                'CLIENT': client['client'],
                'PRODUCT': client['product'],
                'BUREAU': client['bureau'],
                'FORMAT': self.bureau_format[client['bureau']],
                'PULL_DATE': client['pull_date'], 
                'PULL_NAME': client['pull_name'],
                'ME_VERSION': client['me_version']
            }
            node_name, is_optional = table_info['pipeline_input_node'], table_info['optional']
            # Rely on s3 schema manager to infer the path
            s3sm = S3SchemaManager(**self.s3_schema, bureau=client['bureau'])
            s3sm.set_partition(spec)
            table_path = s3sm.get_path()
            is_exist = s3fs.S3FileSystem(anon=False).exists(table_path)
            if not is_optional and not is_exist:
                raise Exception(f'{table_path} is required but not exist')
            if is_exist:
                data[node_name] += [table_path]
    self.add_client_data_to_data_section(data['client_data'], self.client_data_asset)
    return data
```
### For each table in data_table_info, this uses our s3client parameters to resolve the actual S3 path where that table's data lives, verifies it exists (raising an error if a required table is missing), and builds a dict mapping each pipeline input node to its S3 path

In [200]:
target_table = cmb.target_spec['target_table']

In [201]:
target_table

'target'

In [202]:
data_table_info.keys()

dict_keys(['app', 'client_data', 'target', 'trade', 'inquiry', 'collection', 'bankruptcy', 'member', 'authorized', 'non_dedupe_inquiry', 'lexis_nexis', 'trended_noplaceholders_fe', 'corev2_5707_noplaceholders_fe', 'corev2_5708_noplaceholders_fe', 'config'])

In [203]:
s3_schema = load_asset('s3_schema/client_data_schema.json')


In [204]:
s3_schema 


{'name': 'client_data',
 'base': 's3://power-client-data-staging/PROCESSED/DATA/',
 'partitions': ['TABLE',
  'VERSION',
  'CLIENT',
  'PRODUCT',
  'BUREAU',
  'FORMAT',
  'PULL_DATE',
  'PULL_NAME',
  'ME_VERSION']}

In [205]:
bureau_format = {
        "experian": "arf7",
        "transunion": "prama",
        "equifax": "cms_6"
    }

## We can see below that make_client_data is taking the information and organizing our transformers in the same way our graph will be organized but each node having the path where the data is located

In [206]:
client_data_s3 = {info['pipeline_input_node']: [] for _, info in data_table_info.items()}
for client in cmb.config['s3clients']:
    for table_name, table_info in data_table_info.items():
        print(f'table_name: {table_name}')
        if table_name != 'target':
            table = table_info['s3_db']
        else:
            table = target_table
        spec = {
            'TABLE': table,
            'VERSION': client['version'], 
            'CLIENT': client['client'],
            'PRODUCT': client['product'],
            'BUREAU': client['bureau'],
            'FORMAT': bureau_format[client['bureau']],
            'PULL_DATE': client['pull_date'], 
            'PULL_NAME': client['pull_name'],
            'ME_VERSION': client['me_version']
        }
        node_name, is_optional = table_info['pipeline_input_node'], table_info['optional']
        # Rely on s3 schema manager to infer the path
        s3sm = S3SchemaManager(**s3_schema, bureau=client['bureau'])
        s3sm.set_partition(spec)
        table_path = s3sm.get_path()
        print(f'table_path: {table_path}')
        is_exist = s3fs.S3FileSystem(anon=False).exists(table_path)
        if not is_optional and not is_exist:
            raise Exception(f'{table_path} is required but not exist')
        if is_exist:
            client_data_s3[node_name] += [table_path]


table_name: app
table_path: s3://power-client-data-staging/PROCESSED/DATA/TABLE=analysis_data/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/
table_name: client_data
table_path: s3://power-client-data-staging/PROCESSED/DATA/TABLE=analysis_data/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/
table_name: target
table_path: s3://power-client-data-staging/PROCESSED/DATA/TABLE=target/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/
table_name: trade
table_path: s3://power-client-data-staging/PROCE

### make_client_data also calls [add_client_data_to_data_section](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L134) which adds the client-provided app features (like AmountRequested, TotalVehicleValue) as a new `client_data` table in the data section, and sets `is_client` as a categorical on the app table
```python
def add_client_data_to_data_section(self, data_path, client_data_asset):
    """
    Adds 'client_data' table into data_section.
    """
    if client_data_asset:
        self.data_section['client_data'] = {
            'data': data_path,
            'asset': client_data_asset
        }
        self.data_section['app']['asset']['info']['categoricals'] = {
            "is_client": {"True": "True", "False": "False"}
        }
```

Note that `self.data_section` here is the same data section we've been building - it was passed into `S3ClientHandler.__init__` and deep copied. So this is just continuing to add to our data map, now with the client_data table entry.

Also note our function call was:
```python
self.client_handler = S3ClientHandler(
    data_section=data_section, 
    s3clients=self.config['s3clients'], 
    client_data_asset=self.config['client_data_asset'],
    target_table=self.target_spec['target_table']
)
```

So the `client_data_asset` here is the one stored on our ClientModelBuilder config - which is the asset from `configuration_api['client_data']['asset']` that describes how to process AmountRequested and TotalVehicleValue

## We have client data but the data_section doesn't have it, so we make sure it points to it

In [209]:
data_section.keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target', 'config'])

In [211]:
cmb.config['client_data_asset']

{'feature_engineering': 'one_to_one',
 'validator': 'base',
 'table_type': 'app',
 'info': {'key': 'ZEST_KEY',
  'app_date': 'appDate',
  'numerics': ['AmountRequested', 'TotalVehicleValue'],
  'categoricals': [],
  'feature_definitions': {'AmountRequested': 'to be removed',
   'TotalVehicleValue': 'to be removed'},
  'placeholders': {'encoding_type': 'one-hot',
   'placeholders': {'AmountRequested': {'NaN': {'flag': False,
      'val': '9999999999'},
     '0': {'flag': False, 'val': '9999999999'},
     '0.0': {'flag': False, 'val': '9999999999'},
     '0.00': {'flag': False, 'val': '9999999999'}},
    'TotalVehicleValue': {'NaN': {'flag': False, 'val': '0.01'},
     '0': {'flag': False, 'val': '0.01'},
     '0.0': {'flag': False, 'val': '0.01'},
     '0.00': {'flag': False, 'val': '0.01'}}}}},
 'io_params': {'keep_features': ['appDate', 'ZEST_KEY'],
  'drop_duplicates': True}}

In [212]:
cmb.config['client_data_asset']

data_section['client_data'] = {'data': client_data_s3['client_data'], 'asset': cmb.config['client_data_asset']}
data_section['app']['asset']['info']['categoricals'] = {
            "is_client": {"True": "True", "False": "False"}
        }

In [ ]:
## note that we are using clinet data so we
client_data_s3['client_data']

['s3://power-client-data-staging/PROCESSED/DATA/TABLE=analysis_data/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/']

## We then call [_update_data_section_data](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/s3_client_handler.py#L55) - this is a no-op for our S3 case
```python
def _update_data_section_data(self):
    for table in self.data_section:
        data = self.data_section[table]['data']
        if isinstance(data, str) and data.endswith('json'):
            path = data[1:] # get rid of #
            self._update_files(path, self.client_data[table])
    
    if 'keep_keys' in self.data_section['app'].get('io_params', {}):
        print('updating keys')
        self._update_keys()
```

* This method is designed for the dshared storage path where data paths are JSON file references (the `#path.json` pattern). It would open those JSON files and append the new client data paths into them
* For our S3 case, the data values are empty lists `[]`, so the `isinstance(data, str)` check fails on every table and nothing happens

# Now we update our data section with the S3 paths that the client handler resolved
```python
client_data_section = copy.deepcopy(self.client_handler.data_section)
for table, entity in self.client_handler.data_section.items():
    client_data_section[table]['data'] = self.client_handler.client_data.get(table, [])
```

This takes the data section (which the handler already updated with the `client_data` table and `is_client` categorical) and overwrites each table's `data` with the actual S3 paths from `make_client_data`. Our data section now has both the processing instructions (assets) and the locations of the data.

In [215]:
data_section['trade']

{'asset': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
  'AggregationEngine': 'aggregation/fe2/trade.json'},
 'data': [],
 'io_params': {'drop_duplicates': True}}

In [216]:
for table, entity in data_section.items():
    print(f"\n=== {table} ===")
    pprint.pprint(entity)
    ## Point it to the data we found 
    data_section[table]['data'] = client_data_s3[table]
    print(f' After fix')
    pprint.pprint(data_section[table])



=== trade ===
{'asset': {'AggregationEngine': 'aggregation/fe2/trade.json',
           'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}
 After fix
{'asset': {'AggregationEngine': 'aggregation/fe2/trade.json',
           'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
           'fe_version': 2},
 'data': ['s3://power-client-data-staging/PROCESSED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'],
 'io_params': {'drop_duplicates': True}}

=== inq ===
{'asset': {'AggregationEngine': 'aggregation/fe2/inquiry.json',
           'InputNormalizer': 'equifax/cms_6/fe2/inquiry.json',
           'fe_version': 2},
 'data': [],
 'io_params': {'drop_duplicates': True}}
 After fix
{'asset': {'AggregationEngine': 'aggregation/fe2/i

# Finally, we call [normalize_io](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L238) to inject `keep_features` into each table's io_params - telling the data loader which columns to actually keep when loading
```python
client_data_section = normalize_io(client_data_section, self.config['model_config'], 
                                    keep_features=list(schema[self.schema].keys()), 
                                    ignore_sources=['config'])
```

This calls through to the [normalize_io utility](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/utils.py#L30):
```python
def normalize_io(data_section, model_config, keep_features, memory_efficient=False, ignore_sources=None):
    ignore_sources = ignore_sources or []
    data_section['app']['io_params']['keep_features'] = keep_features
    data_section = normalize_target(data_section, NORMALIZE_TARGET_TABLES)
    if 'target_spec' in model_config:
        data_section['target']['io_params']['keep_features'] = ['ZEST_KEY', model_config['target_spec']['target_col']]
        data_section['target']['asset']['info']['col'] = model_config['target_spec']['target_col']
    else:
        data_section['target']['io_params']['keep_features'] = ['ZEST_KEY', model_config['target']]
        data_section['target']['asset']['info']['col'] = model_config['target']
    if 'base_features' in model_config:
        with open(model_config['base_features'], 'r') as infile:
            base_features = json.load(infile)
        for data, features in base_features.items():
            if data not in ignore_sources:
                data_section[data]['io_params']['keep_features'] = features
    for table in data_section.keys():
        if 'io_params' in data_section[table]:
            data_section[table]['io_params']['memory_efficient'] = memory_efficient
    return data_section
```

* For the app table, it sets `keep_features` to the schema columns
* For the target table, it sets `keep_features` to just `ZEST_KEY` and our target column (`final_DQ60_m24`)
* For each bureau table, it reads `keep_features.json` from the national model and sets that table's `keep_features` to only the columns the national model actually uses
* The `config` table is excluded via `ignore_sources=['config']` - it doesn't need column filtering since it generates its own features from scratch
* This is the final step in building our data section - after this, every table knows what data to load, how to process it, and which columns to keep

In [220]:
data_section = normalize_io(data_section, cmb.config['model_config'], keep_features=list(schema[cmb.schema].keys()), ignore_sources=['config'])

# We then call [model_constraint_obj.apply_constraints()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L333) to validate and finalize the model config constraints

Note that `model_constraint_obj` was created during [PowerModelingBase.__init__](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/power_modeling_base.py#L54) which ClientModelBuilder inherits from.

## [ModelConstraints.__init__](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/model_constraints.py#L5-L19) parses all the constraint-related fields out of our config
```python
def __init__(self, config):
    dh = DictHelper(config)
    self.model_config = dh.get_value(['model_config'])
    self.force_drop_is_client = dh.get_value(['model_config', 'force_drop_is_client'], True)
    self.enforce_artifact_validation = dh.get_value(['model_config', 'enforce_artifact_validation'], True)
    self.client_data_asset = dh.get_value(['client_data_asset'], {})
    self.numeric_list = dh.get_value(['client_data_asset', 'info', 'numerics'], [])
    self.categorical_list = dh.get_value(['client_data_asset', 'info', 'categoricals'], [])
    self.exclusion_list = dh.get_value(['model_config', 'exclusion_list'], [])
    self.feature_definition_list = dh.get_value(['model_config', 'feature_definition_list'], [])
    self.client_biv_feature = self._extract_client_biv_feature(dh.get_value(['model_config', 'bivariate_fe_instructions'], []), self.numeric_list)
    self.monotonic_feature_list = self._convert_dict_list_to_list(dh.get_value(['model_config', 'monotonic_constraints_list'], []), 'feature')
    self.define_feature_list = self._convert_dict_list_to_list(dh.get_value(['model_config', 'feature_definition_list'], []), 'feature')
    self.key_factor_feature_list = self._convert_dict_list_to_list(dh.get_value(['model_config', 'key_factor_mapping_list'], []), 'feature')
```

* It extracts the client feature lists (numerics, categoricals), the exclusion list, bivariate feature instructions, monotonic constraints, feature definitions, and key factor mappings from the config we built
* It also identifies which bivariate features are derived from client data (for us, `biv_loan_to_value` since it's computed from `client_data_AmountRequested` and `client_data_TotalVehicleValue`)

## [apply_constraints()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/model_constraints.py#L94-L102) then enforces and validates everything
```python
def apply_constraints(self):
    self._add_is_client_constraint()
    self._append_text_to_client_features()
    if self.enforce_artifact_validation:
        self._validate_monotonic_constraints()
        self._validate_feature_definition()
        self._validate_key_factor_mapping()
    return self.model_config
```

* `_add_is_client_constraint()` - adds `client_data_is_client` to the exclusion list so it can't be used as a model feature
* `_append_text_to_client_features()` - prepends a standard disclaimer prefix to client feature definitions for MRM documentation
* The three validation methods ensure that every client feature (that isn't excluded) has a monotonic constraint, a feature definition, and a key factor mapping - raising an error if any are missing
* Returns the finalized `model_config` which we use going forward

In [221]:
model_config = cmb.model_constraint_obj.apply_constraints()

In [222]:
model_config

{'bivariate_fe_instructions': [{'feature_1': 'client_data_AmountRequested',
   'feature_2': 'client_data_TotalVehicleValue',
   'operation': 'divide',
   'new_feature': 'biv_loan_to_value'}],
 'monotonic_constraints_list': [{'feature': 'biv_loan_to_value',
   'monotonic_constraint': 'positive'}],
 'exclusion_list': [{'feature': 'client_data_AmountRequested',
   'reason': 'used indirectly'},
  {'feature': 'client_data_TotalVehicleValue', 'reason': 'used indirectly'},
  {'feature': 'TotalVehicleValue', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncomeBaseSalary', 'reason': 'Not Used'},
  {'feature': 'AmountRequested', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncome', 'reason': 'Not Used'},
  {'feature': 'VehicleValue', 'reason': 'Not Used'},
  {'feature': 'LoanTerm', 'reason': 'Not Used'},
  {'feature': 'LTV', 'reason': 'Not Used'},
  {'feature': 'DTI', 'reason': 'Not Used'},
  {'feature': 'PTI', 'reason': 'Not Used'},
  {'feature': 'payment_to_income', 'reason': 'Not

In [223]:
# Cell 2: Let's see what changed — is_client should now be in exclusion list
[x for x in model_config['exclusion_list'] if 'is_client' in x.get('feature', '')]

[]

# We then call [_retrieve_data_size](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L334) to count how many training rows we have - this determines which pipeline factory we use
```python
client_data_size = self._retrieve_data_size(data_section, self.target_spec['target_col'], model_config.get('data_split', None))
```
```python
def _retrieve_data_size(self, client_data_section, target_col, data_split=None):
    keep_keys = None
    if client_data_section['app']['io_params'].get('keep_keys', None):
        keep_keys = global_load_data(client_data_section['app']['io_params']['keep_keys'])
    app_data = data_loader(client_data_section['app']['data'], asset={}, keep_keys=keep_keys)
    target_data = data_loader(client_data_section['target']['data'], asset={})
    # Assumption: app and target have ZEST_KEY as the row index
    client_data = app_data.merge(target_data[target_col], left_index=True, right_index=True, how='left')
    client_data_size, _ = client_data.shape
    if data_split and DictHelper(data_split).get_value(['train', 'start_date'], None) and DictHelper(data_split).get_value(['train', 'end_date'], None):
        client_data_size = ((client_data['appDate'] >= data_split['train']['start_date']) & (client_data['appDate'] < data_split['train']['end_date']) & client_data[target_col].notna()).sum()
    return client_data_size
```

* This is the first time we actually load any data - it loads the app and target tables from S3 to count rows
* It merges app and target on ZEST_KEY, then filters down to only rows that fall within the training date window (`2021-07-01` to `2022-11-05`) and have a non-null target value (`final_DQ60_m24`)
* The resulting count determines which pipeline factory we get: too few rows means we skip building a client model entirely

In [224]:

client_data_size = cmb._retrieve_data_size(data_section, cmb.target_spec['target_col'], model_config.get('data_split', None))


severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.




In [225]:
client_data_size

24740

In [226]:
cmb.client_pipeline_config

{'VerySmallClientPipelineFactory': {'num_rows': [0, 3000]},
 'SmallClientPipelineFactory': {'num_rows': [3000, 7000]},
 'MediumClientPipelineFactory': {'num_rows': [7000, 21000]},
 'LargeClientPipelineFactory': {'num_rows': [21000, 10000000]}}

In [227]:
pipeline_name = None
for pipeline, size_bound in cmb.client_pipeline_config.items():
    if size_bound['num_rows'][0] < client_data_size <= size_bound['num_rows'][1]:
        pipeline_name = pipeline

# We then call [_infer_pipeline](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L254) to determine which pipeline factory to use based on our training data size
```python
def _infer_pipeline(self, client_data_size):
    pipeline_name = None
    for pipeline, size_bound in self.client_pipeline_config.items():
        if size_bound['num_rows'][0] < client_data_size <= size_bound['num_rows'][1]:
            pipeline_name = pipeline
    if pipeline_name in ['VerySmallClientPipelineFactory', 'SmallClientPipelineFactory'] or pipeline_name is None:
        return None
    else:
        return pipeline_name
```

We have enough data so we get `LargeClientPipelineFactory`. But note what would happen if our data was too small - `_infer_pipeline` would return `None`, which causes `ClientModelBuilder.run()` to [return None, None](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L339-L340):
```python
if model_config.get('pipeline_factory') is None or model_config.get('pipeline_factory') in ['VerySmallClientPipelineFactory', 'SmallClientPipelineFactory']:
    return None, None
```

That would skip the fold-valid pass and give us a `None` client_model_path, which would hit [this branch](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L565-L569) in `run_combination`:
```python
if client_model_path is None:
    print(f'Skipping client-only model because of too few data samples')
    client_model_path = self.get_national_model_path()
    predictor_config = self.make_client_predictor_input(client_model_path, False)
```

This would prepare the ClientPredictor config to use only the national model - no client model, no ensemble.

In [228]:
pipeline_name

'LargeClientPipelineFactory'

# We then call [_inject_pipeline_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L336) which adds our inferred pipeline factory to the model config (only if one wasn't already set)
```python
model_config = self._inject_pipeline_config(model_config, inferred_pipeline)
```
```python
def _inject_pipeline_config(self, model_config, pipeline):
    if model_config.get("pipeline_factory", None):
        return model_config
    model_config["pipeline_factory"] = pipeline
    return model_config
```

In [229]:
model_config

{'bivariate_fe_instructions': [{'feature_1': 'client_data_AmountRequested',
   'feature_2': 'client_data_TotalVehicleValue',
   'operation': 'divide',
   'new_feature': 'biv_loan_to_value'}],
 'monotonic_constraints_list': [{'feature': 'biv_loan_to_value',
   'monotonic_constraint': 'positive'}],
 'exclusion_list': [{'feature': 'client_data_AmountRequested',
   'reason': 'used indirectly'},
  {'feature': 'client_data_TotalVehicleValue', 'reason': 'used indirectly'},
  {'feature': 'TotalVehicleValue', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncomeBaseSalary', 'reason': 'Not Used'},
  {'feature': 'AmountRequested', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncome', 'reason': 'Not Used'},
  {'feature': 'VehicleValue', 'reason': 'Not Used'},
  {'feature': 'LoanTerm', 'reason': 'Not Used'},
  {'feature': 'LTV', 'reason': 'Not Used'},
  {'feature': 'DTI', 'reason': 'Not Used'},
  {'feature': 'PTI', 'reason': 'Not Used'},
  {'feature': 'payment_to_income', 'reason': 'Not

In [230]:
model_config['pipeline_factory'] = pipeline_name

## We then call [make_mb_asset](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L337) which is inherited from [PowerModelingBase](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/base/power_modeling_base.py#L116-L118)
```python
mb_asset = super().make_mb_asset(data_section, model_config)
```
```python
def make_mb_asset(self, data_section, model_config):
    data_section = self.map_target(data_section)
    return {'data': data_section, "config": model_config}
```

This packages everything into the single dict that gets passed to `build_model()`:
* `mb_asset['data']` - the data section: where to find each table's data, how to process it (assets), and what columns to keep (io_params)
* `mb_asset['config']` - the model config: how to train the model - data splits, pipeline factory, exclusion list, monotonic constraints, bivariate FE instructions, feature definitions, and key factor mappings

Note that `map_target` just ensures the target table key is exactly `"target"` - if it had a different name like `"fraud_target"`, it would rename it. In our case the key is already `"target"` so it's a no-op.

In [244]:
mb_asset = {'data': data_section, 'config': model_config}

# We then call [build_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L342) which is the handoff from our post-sale orchestration into the core model building engine
```python
build_model(mb_asset, self.artifacts_path, manifest=self.manifest)
```

This calls through to [build_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/model_builder/build_model.py#L26):
```python
def build_model(asset, output_location, manifest=None):
    output_system = 's3' if output_location.startswith('s3:') else 'dshared'

    print('parsing model-builder asset')
    state = asset_parser(asset)

    data = state['data']
    config = state.get('config', None) or state.get('asset', None)
    archive_mode = config.get('archive_mode', 'regular')
    fe_version = config.get('fe_version', 1)

    print('Configuring model builder')
    builder = ModelBuilder(data=data, data_split=config['data_split'], archive_mode=archive_mode, fe_version=fe_version)
    builder.configure_run(return_outputs=True)
    
    print('building model')
    artifact_output = builder.generate(input_data=data, input_asset=state['input_asset'], **config)
    
    # Archive results
    if output_system == 'dshared':
        handler = FileSystemHandler()
        prefix = output_location
    else:
        bucket, prefix = global_get_bucket_prefix(output_location)
        handler = S3Handler(bucket_name=bucket)
    data_archiver_object = DataArchiver(
        output_dir=prefix, handler=handler, manifest=manifest, save_manifest=True, extras='allow'
    )
    data_archiver_object.archive(artifact_output)
    
    return builder, artifact_output
```

This is where all the heavy lifting happens:
* `asset_parser(asset)` - takes our `mb_asset` and loads the actual data from S3, constructs the feature engines for each table, and returns a `state` dict with everything ready to run
* `ModelBuilder` - builds the artifact computation graph (feature engineering, data splitting, pipeline fitting, scoring, performance metrics, key factors, etc.)
* `builder.generate()` - executes the entire graph: runs FE on all tables, splits by appDate, fits the XGBoost pipeline on training data, scores all splits, computes AUC/KS, and produces all model artifacts
* `DataArchiver` - writes all the artifacts (pipeline.obj, model.obj, scores, metrics, asset.json, etc.) to our S3 output path

# We can see from the [asset_parser code](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/model_builder/asset_parser.py) that it takes our `mb_asset` and actually loads all the data, resolving the FE2 sub-assets for each table
```python
def asset_parser(asset):
    asset = copy.deepcopy(asset)
    config = asset.get('config') or asset.get('asset')
    
    state = {'data': {}, 
            'input_asset': copy.deepcopy(asset),
            'config': config,
            'cache': {}
            }

    for name, info in asset['data'].items():
        data, table_name = _load_data(info, config, name)
        state['data'][table_name] = data

    cache = asset.get('cache', {}) or {}
    for name, info in cache.items():
        data, table_name = _load_data(info, config, name)
        state['cache'][table_name] = data

    return state
```

* For each table in `mb_asset['data']`, it calls `_load_data` which handles asset loading, FE2 parsing, IO params, and data loading
* The result is a `state` dict where `state['data']` has the loaded DataFrames ready to go, and `state['input_asset']` preserves the original mb_asset for archival

## `_load_data` - loads the asset, applies IO params, and reads the actual parquet data
```python
def _load_data(info, config, name):
    asset_info = info.get('asset', {}) or {}
    io_params = info.get('io_params', {}) or {}
    columns = None
    keep_features, drop_duplicates, keep_keys = _get_io_params(io_params)
    if asset_info:
        asset = load_asset(info['asset'])
        if 'table_name' not in asset.keys():
            asset['table_name'] = name
        table_name = asset['table_name'] 
        # If the asset has a fe_version param == 2, load using FE2 methods
        if asset.get('fe_version') == 2:
            asset = parse_asset(asset, keep_features)
        if keep_features:
            if table_name not in ["app", "target", "sample_weight"]:
                asset['info']['keep_features'] = keep_features 
            if asset['info']['key'] not in keep_features:
                columns = keep_features + [asset['info']['key']]
            else:
                columns = keep_features
    else:
        asset = {}
        table_name = name
    data = data_loader(info['data'], asset=asset, columns=columns, 
                       drop_duplicates=drop_duplicates, keep_keys=keep_keys)
    return data, table_name
```

* Loads the asset JSON and checks if it's FE2 - if so, calls `parse_asset` which resolves the InputNormalizer and AggregationEngine sub-assets
* Uses `keep_features` from io_params to limit which columns get loaded from the parquet files
* For bureau tables (not app/target/sample_weight), injects `keep_features` into the asset so the feature engine knows which output columns to produce
* Finally calls `data_loader` which reads the actual parquet data from the S3 paths

## `data_loader` - reads parquet files from S3 and wraps them in a DataFrameEntity
```python
def data_loader(path, asset=None, columns=None, drop_duplicates=False, keep_keys=None, memory_efficient=False):
    data_path = path
    if isinstance(path, list):
        if all(f.endswith('.parquet') for f in path):
            if (not keep_keys or (keep_keys and columns)) and (not memory_efficient):
                if all(f.startswith('s3://') for f in path):
                    endpoint_url = os.environ.get('AWS_ENDPOINT_URL')
                    s3 = s3fs.S3FileSystem(anon=False, client_kwargs={'endpoint_url': endpoint_url})
                    ds = ParquetDataset(path, filesystem=s3, validate_schema=False, use_legacy_dataset=True)
                else:
                    ds = ParquetDataset(path, validate_schema=False, use_legacy_dataset=True)
                data = _load_parquet_dataset(ds.paths, asset, columns, drop_duplicates, keep_keys)
            else:
                N = 200
                results = []
                for i in range(0, len(path), N):
                    sub_path = path[i:i+N]
                    data = data_loader(sub_path, asset, columns)
                    data = filter_keys(data, asset, keep_keys)
                    results.append(data)
                data = DataFrameEntity(pd.concat(results), asset=asset)
        else:
            results = []
            for f in path:
                results.append(data_loader(f, asset, columns, drop_duplicates, keep_keys))
            data = DataFrameEntity(pd.concat(results), asset=asset)
    elif path.startswith('#') and path.endswith('.json'):
        data_path = data_loader(path[1:], asset, columns)
        data = data_loader(data_path, asset, columns, drop_duplicates, keep_keys)
    else:   
        if path.endswith('.parquet') or path.endswith('/'):
            data = read_parquet(data_path, asset=asset, columns=columns)
        elif path.endswith('.json'):
            data = global_load_data(path)
        elif path.endswith('.obj'):
            data = global_load_data(path)
        else:
            raise ValueError('Unable to load {}'.format(path))
        if keep_keys:
            data = filter_keys(data, asset, keep_keys)
    if drop_duplicates:
        idx_keep = ~data.key.duplicated()
        data = data.loc[idx_keep]
    return data
```

* For our S3 case, `path` is a list of S3 parquet paths - it creates an S3FileSystem, builds a ParquetDataset, and reads them in parallel via `_load_parquet_dataset`
* `columns` limits which columns get read from parquet (the keep_features we set in normalize_io)
* `keep_keys` filters to specific ZEST_KEYs if provided
* `drop_duplicates` deduplicates by ZEST_KEY
* The result is a `DataFrameEntity` - a pandas DataFrame with the asset dict attached as metadata
* Note: `data_loader` is a pure data loading function - it does NOT call `engine_factory` or construct any feature engines. The asset is just attached as metadata. The actual engine construction happens later when `ModelBuilder` builds the ZAML pipeline during `builder.generate()`

## `parse_asset` - resolves and assembles the FE2 sub-assets (for bureau tables only)
```python
def parse_asset(asset, keep_features=None, required_keys=['InputNormalizer', 'AggregationEngine']):
    _validate_FE2_factory_asset(asset, required_keys)
    table_name = asset.get('table_name')
    asset['info'] = {}
    sub_asset_ext_map_sections = {
        'InputNormalizer': ['mapping', 'preprocess'],
        'AggregationEngine': ['postprocess']
    }
    for sub_asset_name, external_mapping_sections in sub_asset_ext_map_sections.items():
        sub_asset = asset.get(sub_asset_name)
        if sub_asset is None:
            continue
        sub_asset = load_asset(sub_asset)
        if 'table_name' not in sub_asset.keys():
            sub_asset['table_name'] = asset.get('table_name')
        sub_asset = process_dict_external_assets(sub_asset)
        sub_asset = process_listed_external_assets(sub_asset, transformer_sections=external_mapping_sections)
        asset.update({sub_asset_name: sub_asset})
    if table_name is None:
        table_name = _validate_FE2_table_name(asset)
        asset['table_name'] = table_name
    dh = DictHelper(asset)
    key = dh.get_value(['InputNormalizer', 'key']) or 'ZEST_KEY'
    asset['info']['key'] = key
    if keep_features:
        if table_name not in ["app", "target", "sample_weight"]:
            if 'AggregationEngine' in asset:
                asset['AggregationEngine']['keep_features'] = keep_features 
            asset['info']['keep_features'] = keep_features
    return asset
```

* This is only called for FE2 bureau tables (trade, inq, collec, bnkr, member)
* It loads and resolves the InputNormalizer and AggregationEngine sub-assets - including any external asset references
* Injects `keep_features` into the AggregationEngine so it knows which output features to produce
* Note: `parse_asset` does NOT call `engine_factory` - it just assembles the asset dict. The actual engine construction happens later when `ModelBuilder` builds the pipeline during `builder.generate()`

## `_get_io_params` - extracts and resolves the IO parameters
```python
def _get_io_params(io_params):
    keep_features = io_params.get('keep_features', None)
    drop_duplicates = io_params.get('drop_duplicates', False)
    keep_keys = io_params.get('keep_keys', False)
    memory_efficient = io_params.get('memory_efficient', False)
    if isinstance(keep_keys, str) and keep_keys.endswith('.json'):
        keep_keys = data_loader(keep_keys)
    if isinstance(keep_features, str) and keep_features.endswith('.json'):
        keep_features = data_loader(keep_features)
    return keep_features, drop_duplicates, keep_keys
```

* Pulls out `keep_features`, `drop_duplicates`, and `keep_keys` from the io_params we set during `normalize_io`
* If `keep_keys` or `keep_features` are file paths (ending in `.json`), it loads them - this is how the national model's `keep_features.json` gets resolved into an actual list of column names

In [247]:
state = asset_parser(mb_asset)

In [248]:
data = state['data']
config = state.get('config', None) or state.get('asset', None) # get config backwards compatible for "asset" as well 



In [250]:
data.keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target', 'config', 'client_data'])

In [251]:
data['trade'].head(5)

,trade_count__all_accounts,trade_months_since_oldest_account_opened__all_accounts,trade_months_since_most_recent_DQ30_or_greater__all_accounts,trade_months_since_most_recent_DQ60_or_greater__all_accounts,trade_percent_never_dq__all_accounts,trade_percent_derog__all_accounts,trade_percent_accounts_with_DQ30_or_greater_in_last_24_months__all_accounts,trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__all_accounts,trade_percent_accounts_with_DQ120_or_greater_in_last_24_months__all_accounts,trade_percent_accounts_with_DQ30_or_greater_in_last_12_months__all_accounts,...,trade_mean_loan_amount__never_delinquent_closed_installment,trade_min_past_due_to_monthly_payment__joint_closed_installment,trade_min_past_due_to_monthly_payment__secure_closed_installment,trade_max_past_due_to_monthly_payment__all_closed_auto,trade_count__non_derog_closed_personal,trade_sum_loan_amount__non_derog_closed_personal,trade_sum_loan_amount__never_delinquent_closed_personal,trade_mean_loan_amount__never_delinquent_closed_personal,trade_count__individual_closed_personal,trade_count__non_derog_closed_unsecure_personal
ZEST_KEY,,,,,,,,,,,,,,,,,,,,,
210623_1_276_12,8.0,483.952443,-4.0,-4.0,1.000000,0.0,0.00,0.00,0.0,0.00,...,-2.000000,-2.0,-2.0,-2.0,0.0,-2.0,-2.0,-2.000000,0.0,0.0
210629_1_276_12,9.0,108.782521,35.0,35.0,0.222222,0.0,0.00,0.00,0.0,0.00,...,-3.000000,-3.0,-3.0,-2.0,0.0,-2.0,-2.0,-2.000000,0.0,0.0
210631_1_276_12,37.0,165.917165,66.0,66.0,0.756757,0.0,0.00,0.00,0.0,0.00,...,10808.400000,-4.0,-4.0,-4.0,1.0,5139.0,5139.0,5139.000000,1.0,1.0
210632_1_276_12,50.0,249.434280,6.0,6.0,0.960000,0.0,0.02,0.02,0.0,0.02,...,4771.545455,-3.0,-4.0,-4.0,0.0,-2.0,-2.0,-2.000000,0.0,0.0
210633_1_276_12,31.0,300.096511,-4.0,-4.0,1.000000,0.0,0.00,0.00,0.0,0.00,...,2666.666667,-3.0,-3.0,-2.0,3.0,8000.0,8000.0,2666.666667,3.0,3.0


In [252]:
data['client_data'].head(5)

,FraudScore,CustomScore2,ChildSupport,ApplicantType,OtherExpense2,MonthlyLiability,EmploymentStatus,CustomScore5,MembershipMonths,benchmark_scores,...,flg_active_co,flg_exist_fe_inquiry,flg_percent_deferred,flg_contains_deferred,VS30,VS40,perf_state_cde,perf_acct_typ_cde,kob,ARCHIVE_DATE
ZEST_KEY,,,,,,,,,,,,,,,,,,,,,
193006_1_276_12,0,789,0,P,0,3656,EM,nan,0,785.0,...,0.0,0,0.0,0,NaN,NaN,Unknown,Unknown,Unknown,2020-12-31
193006_2_276_12,0,784,0,C,0,0,EM,nan,0,764.0,...,0.0,0,0.0,0,NaN,NaN,Unknown,Unknown,Unknown,2020-12-31
193009_2_276_12,0,835,0,C,0,0,SE,nan,0,823.0,...,0.0,0,0.0,0,NaN,NaN,Unknown,Unknown,Unknown,2020-12-31
193009_1_276_12,0,823,0,P,0,1524,EM,nan,0,798.0,...,0.0,0,0.0,0,NaN,NaN,Unknown,Unknown,Unknown,2020-12-31
193010_1_276_12,0,679,0,P,0,218,EM,nan,0,691.0,...,0.0,0,0.0,0,NaN,NaN,Unknown,Unknown,Unknown,2020-12-31


In [249]:
archive_mode = config.get('archive_mode', 'regular')

In [254]:
fe_version = config.get('fe_version', 1)

In [237]:
config

# We then call [build_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/client_model_builder.py#L342) which is the handoff from our post-sale orchestration into the core model building engine
```python
build_model(mb_asset, self.artifacts_path, manifest=self.manifest)
```

This calls through to [build_model](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/model_builder/build_model.py#L26):
```python
def build_model(asset, output_location, manifest=None):
    output_system = 's3' if output_location.startswith('s3:') else 'dshared'

    print('parsing model-builder asset')
    state = asset_parser(asset)

    data = state['data']
    config = state.get('config', None) or state.get('asset', None)
    archive_mode = config.get('archive_mode', 'regular')
    fe_version = config.get('fe_version', 1)

    print('Configuring model builder')
    builder = ModelBuilder(data=data, data_split=config['data_split'], archive_mode=archive_mode, fe_version=fe_version)
    builder.configure_run(return_outputs=True)
    
    print('building model')
    artifact_output = builder.generate(input_data=data, input_asset=state['input_asset'], **config)
    
    # Archive results
    if output_system == 'dshared':
        handler = FileSystemHandler()
        prefix = output_location
    else:
        bucket, prefix = global_get_bucket_prefix(output_location)
        handler = S3Handler(bucket_name=bucket)
    data_archiver_object = DataArchiver(
        output_dir=prefix, handler=handler, manifest=manifest, save_manifest=True, extras='allow'
    )
    data_archiver_object.archive(artifact_output)
    
    return builder, artifact_output
```

## What's happening at a high level

Think of the ModelBuilder like an assembly line blueprint. Before anything runs, it constructs a **directed acyclic graph (DAG)** - a dependency graph where each node is an "artifact" (a unit of work) and the edges say "I need this to be done before I can run."

For example, `ScoresArtifact('train')` needs the fitted pipeline and the training data to exist first. `PerformanceArtifact('train', 'auc')` needs the scores and targets. The DAG encodes all of these dependencies so things execute in the right order.

Here's what happens step by step:

1. **`asset_parser(asset)`** - loads all the parquet data from S3, resolves FE2 sub-assets, and returns a `state` dict with the loaded DataFrames and the config

2. **`ModelBuilder(data=data, data_split=config['data_split'], ...)`** - constructs the DAG via `_build_graph()`. This does NOT run anything yet - it just wires up all the artifact nodes and their dependencies:
```python
def _build_graph(self, input_data, splits, fe_version):
    # Phase 1: Setup artifacts
    default_output = [InputAssetArtifact(), Versions(), MonotonicConstraintsListParser()]
    
    # Phase 2: Data loading artifacts
    helper_artifacts = self._build_data_graph(input_data)
    
    # Phase 3: Split artifacts (train/valid/test by appDate)
    sub_default_output, sub_helper_artifacts = self._build_split_graph(input_data, splits)
    
    # Phase 4: Pipeline fitting
    helper_artifacts.extend([PipeFitter(), PipeFactoryArtifact()])
    
    # Phase 5: Fitted model outputs
    default_output.extend([FittedPipeline(), FittedModel(), TrainHistoryArtifact(), 
                           BestModelParamsArtifact(), FitTimeInfoArtifact()])
    
    # Phase 6: Post-model scoring and evaluation
    post_artifacts, post_helpers = self._build_post_model_graph(splits, fe_version)
    
    # Phase 7: Final persistence
    default_output.extend([StaticAssetArtifact(), MRMPipeline()])
    
    return default_output, helper_artifacts
```

Each artifact is a `DAGNode` (from ZAML's graph library) with `inbounds` (what it depends on) and a `generate()` method (what it actually does). The `ArtifactManager` base class resolves this graph into an execution order using topological sort - so things run in dependency order automatically.

In [255]:
builder = ModelBuilder(data=data, data_split=config['data_split'], archive_mode=archive_mode, fe_version=fe_version)

In [256]:
builder.__dict__.keys()

dict_keys(['archive_mode', 'fe_version', 'shared_params', 'graph', 'output_artifacts', 'input_artifacts', 'name', '_zaml_ver', '_other_params', 'inbounds', 'outbounds'])

## We can see the DAG (directed acyclic graph) that ModelBuilder constructed

* Each node in `builder.graph` represents a single step in the model building process - loading data, splitting, fitting the pipeline, scoring, computing metrics, etc.
* The `inputs` on each node tell us what that step depends on - for example, `ScoresArtifact('train')` depends on the fitted pipeline and the training data
* When we call `builder.generate()`, the ArtifactManager walks this graph in topological order - meaning each node only runs after all of its dependencies have completed


In [259]:
builder.graph.__dict__

{'nodes': [<Artifact name: 'input_asset', class name: 'InputArtifact', inputs: ['input_asset']>,
  <Artifact name: 'input_data', class name: 'InputArtifact', inputs: ['input_data']>,
  <Artifact name: 'monotonic_constraints_list', class name: 'InputArtifact', inputs: ['monotonic_constraints_list']>,
  <Artifact name: 'add_default_monotonic_constraints', class name: 'InputArtifact', inputs: ['add_default_monotonic_constraints']>,
  <Artifact name: 'fe_version', class name: 'InputArtifact', inputs: ['fe_version']>,
  <Artifact name: 'data_split', class name: 'InputArtifact', inputs: ['data_split']>,
  <Artifact name: 'train_sample_weight', class name: 'InputArtifact', inputs: ['train_sample_weight']>,
  <Artifact name: 'valid_sample_weight', class name: 'InputArtifact', inputs: ['valid_sample_weight']>,
  <Artifact name: 'test_sample_weight', class name: 'InputArtifact', inputs: ['test_sample_weight']>,
  <Artifact name: 'n_tiers', class name: 'InputArtifact', inputs: ['n_tiers']>,
  <Ar

## We then call [configure_run](https://github.com/Katlean/zaml/blob/5634992cee256498ec1626e636b1a6197e5df711/zaml/artifact_engine/artifact_manager.py#L257-L313) which tells the ArtifactManager how to handle outputs when it runs
```python
builder.configure_run(return_outputs=True)
```
```python
def configure_run(self, archive_dir=None, return_outputs=True, artifact_errors='raise', 
                  archive_all=False, archive_params=False, table_format='parquet', cache=False):
    self.archivers = []
    if return_outputs:
        self.archivers.append(DictionaryOutput())
    if archive_dir is not None:
        self.archivers.append(FileSystemArchiver(archive_dir, table_format=table_format))
    if len(self.archivers) == 0:
        raise ValueError('At least one of "archive_dir" or "return_outputs" must be valid.')
    if artifact_errors not in ["omit", "raise"]:
        raise ValueError('artifact_errors must be "omit" or "raise".')
    self.artifact_errors = artifact_errors
    self.archive_all = archive_all
    self.archive_params = archive_params
    self.cache = cache
```

* This sets up the "archivers" - the destinations where artifact outputs will go as they are produced

In [260]:
builder.configure_run(return_outputs=True)

# We then call [generate](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/model_builder/build_model.py#L54) which executes the entire model building process
```python
artifact_output = builder.generate(input_data=data, input_asset=state['input_asset'], **config)
```

This calls [ModelBuilder.generate](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/model_builder/model_builder.py#L138-L144) which wraps the parent class with timing:
```python
def generate(self, **inputs):
    start_time = time.time()
    artifact_output = super().generate(**inputs)
    time_spent = time.time() - start_time
    run_time = { 'run_time': '{:.3f}s'.format(time_spent)}
    self._archive_run_time(run_time)
    return artifact_output
```

Which calls [ArtifactManager.generate](https://github.com/Katlean/zaml/blob/5634992cee256498ec1626e636b1a6197e5df711/zaml/artifact_engine/artifact_manager.py#L330-L452) - this is the engine that actually walks the DAG:
```python
def generate(self, **inputs):
    if not hasattr(self, "archivers"):
        self.configure_run()

    self._check_inputs(**inputs)
    self.__reset_is_run_dict()
    optional_artifacts_set = self.__get_optional_artifacts_set()

    self.artifacts_pool = {}
    self.__feed_pool(inputs, self.artifacts_pool)

    for node in self.graph.nodes:
        start_time = time()
        logger.info(f"Executing {type(node).__name__} <{node.name}>...")
        artf = self._generate_single_node(node, optional_artifacts_set)
        if not isinstance(artf, _NoOutput):
            self.artifacts_pool[node.name] = artf
            self._archive_single_node(node, artf)
        if not self.cache:
            self.__clear_pool(node, self.artifacts_pool)
        if node not in self.input_artifacts:
            run_time = time() - start_time
            logger.info(f"Finished <{node.name}>, total time spent: {timedelta(seconds=run_time)}")

    if self.archive_params:
        self._archive_params()

    return self.archivers[0].get_receipt()
```

## What's happening at a high level

* `generate()` walks through every node in the DAG in order, keeping a shared `artifacts_pool` dictionary where each node's output gets stored so downstream nodes can use it
* The DAG handles everything end-to-end: loading and merging data, splitting by appDate, fitting the XGBoost pipeline, scoring all splits, computing AUC/KS, and producing all the model artifacts
* At the end it returns a dictionary of all output artifacts, which `build_model` passes to `DataArchiver` for saving

In [261]:
artifact_output = builder.generate(input_data=data, input_asset=state['input_asset'], **config)

INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <add_default_monotonic_constraints>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <add_default_monotonic_constraints>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <fe_version>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <train_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <train_sample_weight>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <valid_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are a

-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.011s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.003s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.011s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.002s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.011s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spent: 0.003s
-------------------------
Name: member
Transformer type: None
Number of features: 32
Time spent: 0.014s
-------------------------
Name: trade
Transformer type: None
Number of features: 234
Time spent: 0.045s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 1
Time spent: 0.193s
-------------------------
Name: bnkr FE
Transformer type: 

-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 3.086s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.016s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.002s
[0]	train-auc:0.75643	val-auc:0.77009
[100]	train-auc:0.82761	val-auc:0.83609
[200]	train-auc:0.82938	val-auc:0.83403
[300]	train-auc:0.82860	val-auc:0.83161
[306]	train-auc:0.82873	val-auc:0.83161


INFO:zaml.artifact_engine.logger:Finished <pipeline_fitter>, total time spent: 0:01:23.108087
INFO:zaml.artifact_engine.logger:Executing FittedPipeline <pipeline>...
INFO:zaml.artifact_engine.logger:Finished <pipeline>, total time spent: 0:00:00.000302
INFO:zaml.artifact_engine.logger:Executing FitTimeInfoArtifact <fit_time_info>...
INFO:zaml.artifact_engine.logger:Finished <fit_time_info>, total time spent: 0:00:00.000259
INFO:zaml.artifact_engine.logger:Executing PipeFactoryArtifact <pipe_factory>...
INFO:zaml.artifact_engine.logger:Finished <pipe_factory>, total time spent: 0:00:00.000273
INFO:zaml.artifact_engine.logger:Executing FittedModel <model>...
INFO:zaml.artifact_engine.logger:Finished <model>, total time spent: 0:00:00.000255
INFO:zaml.artifact_engine.logger:Executing FeDataArtifact <train_fe_data>...


-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.000s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.000s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.000s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spent: 0.000s
-------------------------
Name: member
Transformer type: None
Number of features: 32
Time spent: 0.000s
-------------------------
Name: trade
Transformer type: None
Number of features: 234
Time spent: 0.000s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 1
Time spent: 0.013s
-------------------------
Name: bnkr FE
Transformer type: 

...


INFO:zaml.artifact_engine.logger:Finished <valid_fe_data>, total time spent: 0:00:20.688824
INFO:zaml.artifact_engine.logger:Executing FeDataArtifact <test_fe_data>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 1.497s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.004s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.001s
-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.000s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.000s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.000s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spe


INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:00:21.920377
INFO:zaml.artifact_engine.logger:Executing StaticAssetArtifact <static_asset>...
INFO:zaml.artifact_engine.logger:Finished <static_asset>, total time spent: 0:00:00.051483
INFO:zaml.artifact_engine.logger:Executing TrainHistoryArtifact <train_history>...
INFO:zaml.artifact_engine.logger:Finished <train_history>, total time spent: 0:00:00.000230
INFO:zaml.artifact_engine.logger:Executing BestModelParamsArtifact <best_model_params>...
INFO:zaml.artifact_engine.logger:Finished <best_model_params>, total time spent: 0:00:00.000433
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <train_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_scores>, total time spent: 0:00:00.087083
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <train_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_submodel_scores>, total time spent: 0:00:00.000546
INFO:zaml.artif

-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 1.503s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.005s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.001s


INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:00.045448
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.000315
INFO:zaml.artifact_engine.logger:Executing CalibrationObjectArtifact <calibration_object>...
INFO:zaml.artifact_engine.logger:Finished <calibration_object>, total time spent: 0:00:00.203280
INFO:zaml.artifact_engine.logger:Executing FeatureDefinition <feature_definition>...

INFO:zaml.artifact_engine.logger:Finished <feature_definition>, total time spent: 0:00:00.045405
INFO:zaml.artifact_engine.logger:Executing KeyFactorMapping <key_factors_mapping>...
INFO:zaml.artifact_engine.logger:Finished <key_factors_mapping>, total time spent: 0:00:00.000563
INFO:zaml.artifact_engine.logger:Executing ValueBasedKeyFactorMapping <value_based_key_factor_mapping>...
INFO:zaml.artifact_engine.logger:Finished <value_based_ke

# Note for our case we will save this to a local path in GYD_Modeling

In [ ]:
GYD_Modeling_Directory = f'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/'

# Since this is the first pass model with the validation set held out, we save it to the supporting model location. The supporting model's main purpose is to find the optimal number of estimators - we'll use that in pass 2 when we fold the validation set into training.

In [306]:
supporting_model_path = os.path.join(GYD_Modeling_Directory, f'CLIENT_ONLY_MODEL_{your_name}/supporting_model/')
os.makedirs(supporting_model_path, exist_ok=True)

In [307]:
output_system = 'dshared' ## use dshared for our case

In [308]:
handler = FileSystemHandler()
prefix = supporting_model_path

In [309]:
from zestio.handler import FileSystemHandler

handler = FileSystemHandler()
manifest_template = load_asset('power/artifact_manifest_template.json')
manifest = DictHelper(manifest_template).replace_value_segment('path', 'data=SOURCE', 'data=client')

data_archiver_object = DataArchiver(
    output_dir=prefix, handler=handler, manifest=manifest, save_manifest=True, extras='allow'
)
data_archiver_object.archive(artifact_output)

# Note that `ClientModelBuilder.run()` returns `(self.artifacts_path, model_config['pipeline_factory'])` - so `client_model_path` is where the artifacts were saved, and `pipeline` is the pipeline factory config (e.g., `'LargeClientPipelineFactory'`), not the model output itself

# We have now saved our supporting model. The next step is to [fold the validation set into training](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L553-L559) and retrain with the optimal number of estimators we just found.

We call [make_fold_valid_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L479) which modifies our client model build config for the second pass:
```python
def make_fold_valid_config(self, pipeline, client_model_path, client_model_build_config):
    if isinstance(pipeline, str):
        all_pipelines = get_default_factory_configs()
        pipeline_spec = all_pipelines[pipeline]
    else:
        pipeline_spec = pipeline
    best_num_estimator = self.get_best_num_iteration(client_model_path)
    pipeline_spec['model']['params'].pop('early_stopping_rounds', None)
    pipeline_spec['model']['params']['n_estimators'] = best_num_estimator
    client_model_build_config['model_config']['pipeline_factory'] = pipeline_spec
    client_model_build_config['model_config']['data_split']['train']['end_date'] = \
        client_model_build_config['model_config']['data_split']['valid']['end_date']
    client_model_build_config['model_config']['data_split'].pop('valid', None)
    client_model_build_config['model_settings'] = \
        self.configuration_api.get('national_model', {}).get('model_settings')
    return client_model_build_config
```

It reads the optimal number of estimators from the supporting model via [get_best_num_iteration](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L180) which loads `best_model_params.json` from the supporting model path:
```python
def get_best_num_iteration(self, client_model_path):
    best_model_params_path = os.path.join(client_model_path, 'best_model_params.json')
    if global_path_exists(best_model_params_path):
        best_model_params = global_load_data(best_model_params_path)
        return best_model_params['best_ntree_limit']
    else:
        raise PowerModelBuilderError(f'{client_model_path} does not contain the best_model_params.json file.')
```

Three key changes happen for pass 2:
* **Pins `n_estimators`** to the best value found during pass 1's early stopping - and removes `early_stopping_rounds` since we no longer need it
* **Extends the training window** to absorb the validation set: train end date moves from `2022-11-05` to `2023-04-21`, and the valid split is removed entirely
* **Carries `model_settings`** through so the mega model config features are still injected

The result is a config where we train on more data with the exact right number of trees - no early stopping needed since we already know the optimum.

# We can see that our best iteration was 106 (0-indexed), so `best_ntree_limit` is 107 - this is the exact number of estimators we'll use for pass 2 with no early stopping

In [310]:
model_config['pipeline_factory']

'LargeClientPipelineFactory'

In [311]:
json.load(open(os.path.join(supporting_model_path, 'best_model_params.json')))

{'best_score': 0.836524028871956,
 'best_iteration': 106,
 'best_ntree_limit': 107}

In [313]:
os.listdir(supporting_model_path)

['test_auc.json',
 'valid_auc.json',
 'splitter.obj',
 'overall_zest_scores',
 'asset.json',
 'fit_time_info.json',
 'value_based_key_factor_mapping.json',
 'feature_definition.parquet',
 'fe_data',
 'train_data_summary.json',
 'artifact_manifest.json',
 'mrm_pipeline.obj',
 'model.obj',
 'parsed_monotonic_constraints_list.json',
 'valid_data_summary.json',
 'test_data_summary.json',
 'train_auc.json',
 'top_features.parquet',
 'app',
 'versions.json',
 'best_model_params.json',
 'valid_ks.json',
 'test_ks.json',
 'score_recalibration_mapping.json',
 'key_factors_mapping.json',
 'feature_importance.parquet',
 'overall_scores',
 'train_ks.json',
 'keep_features.json',
 'model_strategy.json',
 'train_history.json',
 'target',
 'calibration_object.obj',
 'static_asset.json',
 'pipeline.obj']

In [314]:
first_run_pipeline = model_config['pipeline_factory']

In [315]:
first_run_pipeline

'LargeClientPipelineFactory'

In [316]:
all_pipelines = get_default_factory_configs()

In [317]:
pipeline_spec = all_pipelines[first_run_pipeline]
pipeline_spec

{'transformers': [{'zaml_class': 'OneHotEncoder',
   'params': {},
   'fit_params': {'ignore_columns': ['app_is_client']}},
  {'zaml_class': 'FillNA', 'params': {'replace_by': -1, 'add_flags': False}},
  {'zaml_class': 'GeneralFeatureSelection',
   'params': {'features': ['app_is_client']}}],
 'model': {'zaml_class': 'XGBoostModel',
  'params': {'n_estimators': 10000,
   'learning_rate': 0.01,
   'max_depth': 3,
   'subsample': 0.5,
   'scale_pos_weight': 2.5,
   'colsample_bytree': 0.05,
   'min_child_weight': 350,
   'seed': 12,
   'early_stopping_rounds': 200,
   'eval_metric': 'auc',
   'verbose_eval': 100,
   'enable_index_matching': True,
   'tree_method': 'hist'}}}

In [318]:
best_num_estimator = 107
pipeline_spec['model']['params'].pop('early_stopping_rounds', None)
pipeline_spec['model']['params']['n_estimators'] = best_num_estimator

In [319]:
pipeline_spec

{'transformers': [{'zaml_class': 'OneHotEncoder',
   'params': {},
   'fit_params': {'ignore_columns': ['app_is_client']}},
  {'zaml_class': 'FillNA', 'params': {'replace_by': -1, 'add_flags': False}},
  {'zaml_class': 'GeneralFeatureSelection',
   'params': {'features': ['app_is_client']}}],
 'model': {'zaml_class': 'XGBoostModel',
  'params': {'n_estimators': 107,
   'learning_rate': 0.01,
   'max_depth': 3,
   'subsample': 0.5,
   'scale_pos_weight': 2.5,
   'colsample_bytree': 0.05,
   'min_child_weight': 350,
   'seed': 12,
   'eval_metric': 'auc',
   'verbose_eval': 100,
   'enable_index_matching': True,
   'tree_method': 'hist'}}}

# Now we take the client model build config from pass 1 and modify it for pass 2: extend the train end date to absorb the validation period, remove the valid split, carry the model settings through, and update the pipeline factory to use the exact `n_estimators` we found - with early stopping removed

In [350]:
cmb_config_for_valid_in = copy.deepcopy(cmb_config_after_dapt)

In [351]:
cmb_config_for_valid_in

{'clients': ['californiacu_autoloan'],
 's3clients': [{'client': 'californiacu',
   'product': 'autoloan',
   'version': 'v2',
   'bureau': 'equifax',
   'pull_name': '20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu',
   'pull_date': '2026-02-01',
   'me_version': 'v2.1.1'}],
 'storage_location': 's3',
 'client_data_asset': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_type': 'app',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'numerics': ['AmountRequested', 'TotalVehicleValue'],
   'categoricals': [],
   'feature_definitions': {'AmountRequested': 'to be removed',
    'TotalVehicleValue': 'to be removed'},
   'placeholders': {'encoding_type': 'one-hot',
    'placeholders': {'AmountRequested': {'NaN': {'flag': False,
       'val': '9999999999'},
      '0': {'flag': False, 'val': '9999999999'},
      '0.0': {'flag': False, 'val': '9999999999'},
      '0.00': {'flag': False, 'val': '9999999999'}},
     'TotalV

In [352]:
cmb_config_for_valid_in['model_config']['pipeline_factory'] = pipeline_spec

In [353]:
cmb_config_for_valid_in['model_config']['data_split']['train']['end_date'] = cmb_config_for_valid_in['model_config']['data_split']['valid']['end_date']

In [354]:
cmb_config_for_valid_in['model_config']['data_split']

{'train': {'start_date': '2021-07-01', 'end_date': '2023-04-21'},
 'valid': {'start_date': '2022-11-05', 'end_date': '2023-04-21'},
 'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}}

In [355]:
cmb_config_for_valid_in['model_config']['data_split'].pop('valid', None)

{'start_date': '2022-11-05', 'end_date': '2023-04-21'}

In [356]:
cmb_config_for_valid_in['model_config']['data_split']

{'train': {'start_date': '2021-07-01', 'end_date': '2023-04-21'},
 'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}}

In [357]:
cmb_config_for_valid_in['model_settings']

{'product': 'autoloan', 'region': 'west', 'KOB': 'creditunion'}

In [358]:
## this code is not neccesary for our case but would be if we did not already fold in 
cmb_config_for_valid_in['model_settings'] = configuration_api.get('national_model', {}).get('model_settings')

# Now we can call build client model again. It will this time save it to the non-supporting model path

In [359]:
fold_in_model_path = os.path.join(GYD_Modeling_Directory, f'CLIENT_ONLY_MODEL_{your_name}')

In [360]:
ls /d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/

CLIENT_ONLY_MODEL_gazin/


In [361]:
supporting_model_path = os.path.join(GYD_Modeling_Directory, f'CLIENT_ONLY_MODEL_{your_name}')

In [362]:
cmb_fold_in = ClientModelBuilder(cmb_config_for_valid_in, fold_in_model_path)

In [363]:
fold_in_model_path

'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/CLIENT_ONLY_MODEL_gazin'

# Will be exactly the same as before except the n_estimators will be pinned to 107 and no early stopping

In [364]:
client_model_fold_in_path, pipeline_client_model_fold_in = cmb_fold_in.run()

parsing model-builder asset
Configuring model builder


INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <add_default_monotonic_constraints>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <add_default_monotonic_constraints>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <fe_version>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <train_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <train_sample_weight>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <test_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are av

building model


INFO:zaml.artifact_engine.logger:Finished <parsed_monotonic_constraints_list>, total time spent: 0:00:00.875087
INFO:zaml.artifact_engine.logger:Executing SplitterArtifact <splitter>...
INFO:zaml.artifact_engine.logger:Finished <splitter>, total time spent: 0:00:00.000545
INFO:zaml.artifact_engine.logger:Executing DataArtifact <data>...
INFO:zaml.artifact_engine.logger:Finished <data>, total time spent: 0:00:00.000226
INFO:zaml.artifact_engine.logger:Executing ExclusionListParser <parsed_exclusion_list>...
INFO:zaml.artifact_engine.logger:Finished <parsed_exclusion_list>, total time spent: 0:00:00.000423
INFO:zaml.artifact_engine.logger:Executing BivariateFeParser <parsed_biv_fe_instructions>...
INFO:zaml.artifact_engine.logger:Finished <parsed_biv_fe_instructions>, total time spent: 0:00:00.000212
INFO:zaml.artifact_engine.logger:Executing NeutralizationListParser <parsed_neutralization_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifa

-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.012s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.003s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.011s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.003s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.011s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spent: 0.003s
-------------------------
Name: member
Transformer type: None
Number of features: 32
Time spent: 0.014s
-------------------------
Name: trade
Transformer type: None
Number of features: 234
Time spent: 0.047s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 1
Time spent: 0.185s
-------------------------
Name: bnkr FE
Transformer type: 

...


INFO:zaml.artifact_engine.logger:Finished <train_fe_data>, total time spent: 0:00:18.133895
INFO:zaml.artifact_engine.logger:Executing FeDataArtifact <test_fe_data>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 1.617s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.015s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.001s
-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.000s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.000s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.000s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spe


INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:00:19.269628
INFO:zaml.artifact_engine.logger:Executing StaticAssetArtifact <static_asset>...
INFO:zaml.artifact_engine.logger:Finished <static_asset>, total time spent: 0:00:00.051910
INFO:zaml.artifact_engine.logger:Executing TrainHistoryArtifact <train_history>...
INFO:zaml.artifact_engine.logger:Finished <train_history>, total time spent: 0:00:00.000305
INFO:zaml.artifact_engine.logger:Executing BestModelParamsArtifact <best_model_params>...
INFO:zaml.artifact_engine.logger:Finished <best_model_params>, total time spent: 0:00:00.000211
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <train_scores>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 1.495s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.003s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.001s


INFO:zaml.artifact_engine.logger:Finished <train_scores>, total time spent: 0:00:00.147532
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <train_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_submodel_scores>, total time spent: 0:00:00.000430
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <test_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:00.058816
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.001194
INFO:zaml.artifact_engine.logger:Executing CalibrationObjectArtifact <calibration_object>...
INFO:zaml.artifact_engine.logger:Finished <calibration_object>, total time spent: 0:00:00.294489
INFO:zaml.artifact_engine.logger:Executing FeatureDefinition <feature_definition>...

INFO:zaml.artifact_engine.logger:Finished <feature_definition>, total time spent: 0:

In [365]:
client_model_fold_in_path

'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/CLIENT_ONLY_MODEL_gazin'

# We now make the client predictor config - this is the map that tells us how to build the final ensemble model

We call [make_client_predictor_input](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L263) [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L574):
```python
predictor_config = self.make_client_predictor_input(client_model_path, True)
```

Calling with `submodels_need=True` means we want an ensemble - the client model as the baseline plus the national model as a submodel. If we had called with `False` (which happens when the client data was too small to build a client model), it would use only the national model with no ensemble.
```python
def make_client_predictor_input(self, client_model_path, submodels_need=True):
    result = {'fe_version': self.fe_version, 'clients': None, 'baseline_model': None, 
              'submodels': None, 'client_data_split': None, 'target_spec': None, 
              'bureau': None, 'model_splits': None, 'model_alias': None}
    
    result['clients'] = self.__make_clients_for_client_predictor_config()
    result['baseline_model'] = client_model_path
    
    if submodels_need:
        result['data_alias'] = {
            "baseline_model": {"train": "client", "valid": "client", "test": "client"},
            "submodels": [{"train": "national", "valid": "national", "test": "client"}]
        }
        result['model_alias'] = {'baseline_model': 'client_model', 'submodels': ['national_model']}
        result['submodels'] = [self.get_national_model_path()]
    else:
        result['data_alias'] = {"baseline_model": {"train": "national", "valid": "national", "test": "client"}}
        result['model_alias'] = {'baseline_model': 'national_model'}
        result['submodels'] = []
    
    existing_model_path = self.configuration_api.get('client_model', {}).get('existing_model_path', None)
    if existing_model_path:
        asset = global_load_data(os.path.join(existing_model_path, 'asset.json'))
        test_period = asset['config']['data_split']['test']
        if 'target_spec' in asset['config']:
            target_spec = asset['config']['target_spec']
        else:
            target_spec = {'target_table': 'target', 'target_col': asset['config']['target']}
        model_split = ['train', 'valid'] if 'valid' in asset['config']['data_split'] else ['train']
        keep_keys = asset['data']['app']['io_params'].get('keep_keys', None)
    else:
        test_period = self.configuration_api['client_model']['model_config']['data_split']['test']
        target = DictHelper(self.configuration_api).get_value(['client_model', 'model_config', 'target']) or \
                 DictHelper(self.configuration_api).get_value(['client_model', 'model_config', 'target_spec', 'target_col'])
        if DictHelper(self.configuration_api).get_value(['client_model', 'model_config', 'target_spec']):
            target_spec = DictHelper(self.configuration_api).get_value(['client_model', 'model_config', 'target_spec'])
        else:
            target_spec = {'target_table': 'target', 'target_col': target}
        model_split = ['train'] if self.get_fold_valid() else ['train', 'valid']
        keep_keys = self.configuration_api['client_data']['asset']['io_params'].get('keep_keys', None)
    
    result['client_data_split'] = {'test': test_period}
    result['target_spec'] = target_spec
    result['bureau'] = self.get_bureau()
    result['io_params'] = {'keep_keys': keep_keys}
    if keep_keys:
        self.archive_config(global_load_data(keep_keys), 'keep_keys.json', self.final_model_output_path)
    result['model_splits'] = model_split
    
    if self.configuration_api.get('national_model', {}).get('model_settings'):
        new_asset = load_asset('megamodel/config.json')
        new_asset['info']['constant_categorical_feature'] = self.configuration_api['national_model']['model_settings']
        new_asset['info']['placeholders']['placeholders'] = self._make_placeholders_mega_model()
        result['update_engine_assets'] = {}
        result['update_engine_assets']['config'] = new_asset
    
    return result
```

This builds a config dict with:
* **`baseline_model`** - path to our pass 2 client model
* **`submodels`** - list containing the national mega model path (since `submodels_need=True`)
* **`data_alias`** - tells the predictor which data each model should use: the client model uses client data for all splits, the national model uses national data for train/valid but client data for test
* **`model_alias`** - human-readable names: `'client_model'` and `'national_model'`
* **`client_data_split`** - only the test period (`2023-04-21` to `2023-11-01`) since we're evaluating the ensemble on held-out data
* **`model_splits`** - `['train']` because we folded validation in (no valid split exists in pass 2)
* **`update_engine_assets`** - the freshly-built mega model config asset with `constant_categorical_feature` and `placeholders`, so the national model's config table gets the correct autoloan/west/creditunion injection at scoring time

The client S3 paths are built by [__make_clients_for_client_predictor_config](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L237):
```python
def __make_clients_for_client_predictor_config(self):
    s3_schema = load_asset('s3_schema/client_data_schema.json')
    dh = DictHelper(self.configuration_api)
    if dh.get_value(['client_data', 'storage_location'], 'dshared') == 'dshared':
        clients = []
        for client in self.configuration_api['client_data']['clients']:
            clients.append(os.path.join(self.client_db, client))
    else:
        clients = []
        for client in self.configuration_api['client_data']['s3clients']:
            spec = {
                'VERSION': client['version'], 
                'CLIENT': client['client'],
                'PRODUCT': client['product'],
                'BUREAU': client['bureau'],
                'FORMAT': self.bureau_format[client['bureau']],
                'PULL_DATE': client['pull_date'], 
                'PULL_NAME': client['pull_name'],
                "ME_VERSION": client['me_version']
            }
            s3sm = S3SchemaManager(**s3_schema, bureau=client['bureau'])
            s3sm.set_partition(spec)
            clients.append(s3sm.get_path_pattern())
    return clients
```

This uses the same S3 schema manager we saw in `S3ClientHandler` to resolve the client data path pattern - but this time it returns a pattern (not individual table paths) since `ClientPredictor` handles its own data loading internally.

In [366]:
result = {'fe_version': fe_version, 'clients': None, 'baseline_model': None, 
            'submodels': None, 'client_data_split': None, 'target_spec': None, 
            'bureau': None, 'model_splits': None, 'model_alias': None}

In [367]:
## making client path 
s3_schema = load_asset('s3_schema/client_data_schema.json')
dh = DictHelper(configuration_api)

## location is s3

clients = []
for client in configuration_api['client_data']['s3clients']:
    spec = {
        'VERSION': client['version'], 
        'CLIENT': client['client'],
        'PRODUCT': client['product'],
        'BUREAU': client['bureau'],
        'FORMAT': bureau_format[client['bureau']],
        'PULL_DATE': client['pull_date'], 
        'PULL_NAME': client['pull_name'],
        "ME_VERSION": client['me_version']
    }
    s3sm = S3SchemaManager(**s3_schema, bureau=client['bureau'])
    s3sm.set_partition(spec)
    clients.append(s3sm.get_path_pattern())

In [368]:
clients

['s3://power-client-data-staging/PROCESSED/DATA/TABLE=*/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/']

In [ ]:
configuration_api.get('client_model', {}).get('existing_model_path', None)

In [371]:
final_model_path = f'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/final_model_{your_name}'

In [372]:
final_model_path

'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/final_model_gazin'

In [ ]:
result['clients'] = clients
result['baseline_model'] = client_model_fold_in_path
submodels_need = True

if submodels_need:
    result['data_alias'] = {
        "baseline_model": {"train": "client", "valid": "client", "test": "client"},
        "submodels": [{"train": "national", "valid": "national", "test": "client"}]
    }
    result['model_alias'] = {'baseline_model': 'client_model', 'submodels': ['national_model']}
    result['submodels'] = [correct_national_model_path]
else:
    result['data_alias'] = {"baseline_model": {"train": "national", "valid": "national", "test": "client"}}
    result['model_alias'] = {'baseline_model': 'national_model'}
    result['submodels'] = []

existing_model_path = configuration_api.get('client_model', {}).get('existing_model_path', None)
if existing_model_path:
    asset = global_load_data(os.path.join(existing_model_path, 'asset.json'))
    test_period = asset['config']['data_split']['test']
    if 'target_spec' in asset['config']:
        target_spec = asset['config']['target_spec']
    else:
        target_spec = {'target_table': 'target', 'target_col': asset['config']['target']}
    model_split = ['train', 'valid'] if 'valid' in asset['config']['data_split'] else ['train']
    keep_keys = asset['data']['app']['io_params'].get('keep_keys', None)
else:
    test_period = configuration_api['client_model']['model_config']['data_split']['test']
    target = DictHelper(configuration_api).get_value(['client_model', 'model_config', 'target']) or \
             DictHelper(configuration_api).get_value(['client_model', 'model_config', 'target_spec', 'target_col'])
    if DictHelper(configuration_api).get_value(['client_model', 'model_config', 'target_spec']):
        target_spec = DictHelper(configuration_api).get_value(['client_model', 'model_config', 'target_spec'])
    else:
        target_spec = {
            'target_table': 'target',
            'target_col': DictHelper(configuration_api).get_value(['client_model', 'model_config', 'target'])
        }
    fold_valid = configuration_api['client_model']['model_config'].get('fold_valid', True)
    model_split = ['train'] if fold_valid else ['train', 'valid']
    keep_keys = configuration_api['client_data']['asset']['io_params'].get('keep_keys', None)

result['client_data_split'] = {'test': test_period}
result['target_spec'] = target_spec
result['bureau'] = configuration_api['national_model']['model_config']['bureau']
result['io_params'] = {'keep_keys': keep_keys}
if keep_keys:
    global_archive_data(os.path.join(final_model_path, 'keep_keys.json'), global_load_data(keep_keys))
result['model_splits'] = model_split

if configuration_api.get('national_model', {}).get('model_settings'):
    new_asset = load_asset('megamodel/config.json')
    new_asset['info']['constant_categorical_feature'] = configuration_api['national_model']['model_settings']
    new_asset['info']['placeholders']['placeholders'] = placeholders  ## built earlier with _make_placeholders_mega_model logic
    result['update_engine_assets'] = {}
    ## points again to the modeling settings 
    result['update_engine_assets']['config'] = new_asset

In [377]:
result.keys()

dict_keys(['fe_version', 'clients', 'baseline_model', 'submodels', 'client_data_split', 'target_spec', 'bureau', 'model_splits', 'model_alias', 'data_alias', 'io_params', 'update_engine_assets'])

In [390]:
results_we_run = copy.deepcopy(result)

# Client path tells us where the data is located

In [378]:
result['clients']

['s3://power-client-data-staging/PROCESSED/DATA/TABLE=*/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/']

In [379]:
result['baseline_model']

'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/CLIENT_ONLY_MODEL_gazin'

In [380]:
result['submodels']

['/d/shared/national_model_warehouse/national_models/fe2/VERSION=211/BUREAU=mega/PRODUCT=mega/FEATURE=v2_member/ITERATION=1']

In [381]:
result['client_data_split']

{'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}}

In [382]:
result['target_spec']

{'target_table': 'target', 'target_col': 'final_DQ60_m24'}

In [383]:
result['bureau']

'equifax'

In [384]:
result['model_splits']

['train']

In [385]:
result['model_alias']

{'baseline_model': 'client_model', 'submodels': ['national_model']}

In [386]:
result['data_alias']

{'baseline_model': {'train': 'client', 'valid': 'client', 'test': 'client'},
 'submodels': [{'train': 'national', 'valid': 'national', 'test': 'client'}]}

In [387]:
result['io_params']

{'keep_keys': None}

In [388]:
result['update_engine_assets']

{'config': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_name': 'config',
  'table_type': 'one_to_one',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'categoricals': {},
   'constant_categorical_feature': {'product': 'autoloan',
    'region': 'west',
    'KOB': 'creditunion'},
   'placeholders': {'encoding_type': 'one-hot',
    'drop_original_features': True,
    'placeholders': {'KOB': {'creditunion': {'flag': True,
       'val': 'creditunion'},
      'regionalbank': {'flag': True, 'val': 'regionalbank'}},
     'product': {'autoloan': {'flag': True, 'val': 'autoloan'},
      'creditcard': {'flag': True, 'val': 'creditcard'},
      'personalloan': {'flag': True, 'val': 'personalloan'},
      'homeequity': {'flag': True, 'val': 'homeequity'},
      'businessloan': {'flag': True, 'val': 'businessloan'}},
     'region': {'west': {'flag': True, 'val': 'west'},
      'northeast': {'flag': True, 'val': 'northeast'},
      'south': {'flag': True, 'val': 's

# We now call [run_client_predictor](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L584) which builds our final ensemble model

The [source code](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/power/post_sale/power_model_builder.py#L514):
```python
def run_client_predictor(self, predictor_input):
    cp = ClientPredictor(predictor_input, self.final_model_output_path)
    cp.run()
```

## Step 1: [ClientPredictor.__init__](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L61-L84) - parses the predictor config
```python
def __init__(self, conf, outpath):
    self.conf = conf
    self.outpath = outpath
    conf_dh = DictHelper(conf)
    self.baseline_model_path = conf['baseline_model']
    self.client_paths = conf.get('clients', None)
    self.national_paths = DictHelper(conf).get_value(['national_samples','data'], None)
    self.target_spec = conf.get('target_spec', None) or {'target_table': 'target', 'target_col': conf['target']}
    self.submodels = conf.get('submodels', None)
    self.client_data_split = conf['client_data_split']
    self.model_data_split = conf.get('model_splits', ['train'])
    self.keep_keys = conf_dh.get_value(['io_params','keep_keys'], None)
    self.model_max_rows = conf.get('model_max_rows', 1000000)
    if self.model_max_rows == 'all':
        self.model_max_rows = None
    self.data_alias = conf.get('data_alias')
    self._data_name_mapping = self._get_data_name_mapping(self.data_alias)
    self.data_list = ['app', 'target', 'fe_data', 'scores', 'zest_scores', 'submodel_scores']
    self.split_tags = set(list(conf['client_data_split'].keys()) + conf['model_splits'])
    self.validate_arguments()
```

* Extracts the baseline model path (our pass 2 client model), submodels (the national mega model), client S3 paths, data splits, and aliases from the config we built

## Step 2: [ClientPredictor.run()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L175-L184)
```python
def run(self):
    self._archive_config(self.conf, 'client_predictor_config.json', self.outpath)
    asset = self.build_asset()
    state = self.load_state(asset)
    graph = self.create_graph(state)
    output = self.excute_graph(graph, state)
    data_to_archive, manifest = self.make_archive_data(output)
    self._archive_artifact(self.outpath, data_to_archive, manifest)
    self._archive_config(asset, 'asset.json', self.outpath)
    return output
```

This follows the same pattern as `build_model` - build an asset, load state, construct a DAG, execute it, archive results. Let's trace each step:

## Step 3: [build_asset()](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L108) → [AssetBuilder](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/asset.py#L22-L50)
```python
def build_asset(self):
    asset_builder = AssetBuilder(self.conf)
    return asset_builder.get_asset()
```
```python
class AssetBuilder:
    def __init__(self, conf):
        self.conf = conf
        conf_helper = DictHelper(self.conf)
        self.baseline_model_path = conf_helper.get_value(['baseline_model'])
        self.client_data_paths = conf_helper.get_value(['clients'])
        self.target_spec = conf_helper.get_value(['target_spec']) or {'target_table': 'target', 'target_col': conf_helper.get_value(['target'])}
        self.national_data_section = conf_helper.get_value(['national_samples', 'data'])
        self.national_data_schema = conf_helper.get_value(['national_samples', 'schema'])
        self.keep_keys = conf_helper.get_value(['io_params', 'keep_keys'])
        self.target_col = self.target_spec['target_col']
        self.submodels = conf_helper.get_value(['submodels'])
        self.update_engine_assets = conf_helper.get_value(['update_engine_assets'])
        
        self.asset, self.data_section = self._get_asset_from_baseline_model(self.baseline_model_path)
        
        if self.client_data_paths:
            self.asset = self._add_client_data(self.asset)
        if self.national_data_section:
            self.asset = self._add_national_data(self.asset)
        self.asset = self._add_baseline_model_to_cache(self.asset)
        
        if self.submodels:
            self.asset = self._add_submodels_to_cache(self.asset)
        
        self.asset = self._verify_submodel_keep_features(self.asset, self.baseline_model_path, self.submodels)
```

AssetBuilder does the following:
* **`_get_asset_from_baseline_model`** - loads `asset.json` from our pass 2 client model, then calls `_update_assets` which replaces the config table's asset with our `update_engine_assets` (the mega model config with `constant_categorical_feature` and placeholders). This ensures the national model will use the correct autoloan/west/creditunion config flags
* **`_add_client_data`** - wires in the client S3 data paths for each table (trade, inq, app, target, etc.) so the ensemble can score on client data
* **`_add_baseline_model_to_cache`** - loads the client model's pipeline.obj, model.obj, splitter, train/valid app, target, and fe_data into the asset's `cache` section
* **`_add_submodels_to_cache`** - does the same for the national mega model - loads its pipeline.obj, model.obj, and training artifacts into cache

## Step 4: [load_state(asset)](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L111) → [State](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/state.py#L9-L34)
```python
def load_state(self, asset):
    return State(asset, self.conf, self.submodels, self.model_max_rows, self.data_alias).data
```
```python
class State:
    def __init__(self, asset, config, submodels=None, max_nrows=None, data_name_mapping=None):
        self.asset = asset
        self.config = config
        self.submodels = submodels
        self.max_nrows = max_nrows
        self._data_name_mapping = self._get_data_name_mapping(data_name_mapping)
        self._state = self._load_state(self.asset, self.submodels)
        self._state = self._add_config_to_state(self._state, self.config)
        self._state = self._update_engine_assets(self._state, self.config)
        self._submodel_names = None
        if self.max_nrows:
            self._state = self.sample_split(self._state)
```

* **`_load_state`** - calls `asset_parser` to load all the data and cache items (both models' artifacts) into a state dict. Groups submodel artifacts together under `submodel_0_artifacts`
* **`_add_config_to_state`** - adds the feature definition list, key factor mapping list, and zest score mapping to the state cache
* **`_update_engine_assets`** - this is where `engine_factory` finally gets called. For each asset in `update_engine_assets` (our config table), it constructs a feature engine and replaces the corresponding transformer in the baseline model's pipeline. This ensures the client model's pipeline uses the correct mega model config flags when scoring
* **`sample_split`** - if `max_nrows` is set (default 1,000,000), downsamples the training data to that limit

## Step 5: [create_graph(state)](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L114-L118) → [ClientPredictorGraph](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/graph.py#L23-L55)
```python
def create_graph(self, state):
    submodel_names = [f'submodel_{i}' for i in range(len(self.submodels))] if self.submodels else None
    graph = ClientPredictorGraph(state, self.model_data_split, self.client_data_split, submodel_names, data_name_mapping=self._data_name_mapping)
    graph.configure_run(return_outputs=True)
    return graph
```
```python
class ClientPredictorGraph(ArtifactManager):
    def __init__(self, state, model_data_splits, client_data_split, submodels=None, verbose=True, data_name_mapping=None):
        self._data_name_mapping = data_name_mapping
        dh = DictHelper(state)
        self.fe_version = dh.get_value(['config', 'fe_version']) or 1
        outputs, helper_artifacts = self._build_graph(state, model_data_splits, client_data_split, submodels)
        super().__init__(outputs, artifact_objects=helper_artifacts, verbose=verbose)

    def _build_graph(self, state, model_data_splits, client_data_split, submodels=None):
        all_splits = set(model_data_splits).union(set(client_data_split))
        default_output = [DataDescription(submodels), UnfoldDataDescription(submodels), Versions()]
        helper_artifacts = self._build_data_graph(input_data=state['data'])
        sub_default_output, sub_helper_artifacts = self._build_split_graph(state['data'], client_data_split)
        default_output.extend(sub_default_output)
        helper_artifacts.extend(sub_helper_artifacts)
        default_existing_artifacts, helper_existing_artifacts = self._existing_model_artifacts(model_data_splits, submodels)
        default_output.extend(default_existing_artifacts)
        helper_artifacts.extend(helper_existing_artifacts)
        post_artifacts, post_helpers = self._build_post_model_graph(all_splits, model_data_splits)
        default_output.extend(post_artifacts)
        helper_artifacts.extend(post_helpers)
        return default_output, helper_artifacts
```

Just like `ModelBuilder`, this constructs a DAG - but this one is designed for ensemble prediction rather than training. The key differences:
* **`_build_data_graph`** - loads client data (app, target, bureau tables)
* **`_build_split_graph`** - splits client data by the test period (`2023-04-21` to `2023-11-01`)
* **`_existing_model_artifacts`** - retrieves the pre-trained artifacts from both models (client + national): their pipelines, train/valid app, fe_data, targets. `NewModel` and `NewPipeline` handle combining the two models into an ensemble
* **`_build_post_model_graph`** - scores the client test data through both models, produces ensemble scores, computes AUC/KS on each split, generates calibrated scores, feature importance, feature definitions, key factors, and pipeline clone/adapt artifacts for cross-bureau scoring

## Step 6: [excute_graph(graph, state)](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/client_predictor/client_predictor.py#L120)
```python
def excute_graph(self, graph, state):
    return graph.generate(input_data=state['data'], **state['data'], **state['cache'], 
                         data_split=self.client_data_split, data_dict={})
```

* This feeds everything into the DAG and executes it - the same `ArtifactManager.generate()` pattern we saw with `ModelBuilder`
* `state['data']` contains the loaded client data tables, `state['cache']` contains both models' pipelines, training artifacts, and config
* The DAG walks in topological order: loads data → splits by test period → runs both pipelines on client test data → combines scores into an ensemble → computes metrics → produces all final artifacts

## Step 7: Archive results
```python
data_to_archive, manifest = self.make_archive_data(output)
self._archive_artifact(self.outpath, data_to_archive, manifest)
self._archive_config(asset, 'asset.json', self.outpath)
```

* `make_archive_data` organizes the output artifacts into a partition structure (scores/split=test/data=client, etc.)
* `_archive_artifact` uses `DataArchiver` to write everything to the final model output path
* The combined `asset.json` (with both models' info and the update_engine_assets) is saved for reproducibility

In [392]:
cp = ClientPredictor(result,final_model_path)

In [393]:
cp.run()

                      but the loaded object is built on ZAML None. Loading serialized
                      object from different versions are not recommended and may yield unexpected
                      errors or results.

                      (name, curr_ver, org_ver): [('graphviz', '0.21', '0.20.3'), ('pg8000', '1.31.5', '1.31.2'), ('simplejson', '3.20.2', '3.20.1'), ('sphinx-rtd-theme', '3.1.0', '3.0.2'), ('sqlalchemy_utils', '0.42.1', '0.41.2'), ('tqdm', '4.67.3', '4.67.1')]. This might yield unexpected result if the object relies
                      on this package.

                      but the loaded object is built on ZAML None. Loading serialized
                      object from different versions are not recommended and may yield unexpected
                      errors or results.



Sampling down to 1000000
Sampling down to 1000000


INFO:zaml.artifact_engine.logger:Executing InputArtifact <baseline_model_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <client_predictor_config>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <baseline_model_supporting_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <submodel_0>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <test_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <test_sample_weight>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <baseline_model_model>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <baseline_model_pipeline>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <baseline_model_train_app>...
INFO:zaml.artifact_engine.logger:Executin

graph fe: 2


INFO:zaml.artifact_engine.logger:Finished <data_description>, total time spent: 0:00:00.808089
INFO:zaml.artifact_engine.logger:Executing UnfoldDataDescription <unfold_data_description>...
INFO:zaml.artifact_engine.logger:Finished <unfold_data_description>, total time spent: 0:00:00.627714
INFO:zaml.artifact_engine.logger:Executing Versions <versions>...
INFO:zaml.artifact_engine.logger:Finished <versions>, total time spent: 0:00:00.000370
INFO:zaml.artifact_engine.logger:Executing ClientSplitterArtifact <splitter>...
INFO:zaml.artifact_engine.logger:Finished <splitter>, total time spent: 0:00:00.000373
INFO:zaml.artifact_engine.logger:Executing NewModel <model>...
INFO:zaml.artifact_engine.logger:Finished <model>, total time spent: 0:00:00.000304
INFO:zaml.artifact_engine.logger:Executing CombineDataArtifact <train_app>...
INFO:zaml.artifact_engine.logger:Finished <train_app>, total time spent: 0:00:00.188386
INFO:zaml.artifact_engine.logger:Executing CombineDataArtifact <train_fe_dat

-------------------------
Name: app
Transformer type: None
Number of features: 18
Time spent: 0.000s
-------------------------
Name: bnkr
Transformer type: None
Number of features: 8
Time spent: 0.000s
-------------------------
Name: client_data
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: collec
Transformer type: None
Number of features: 5
Time spent: 0.000s
-------------------------
Name: config
Transformer type: None
Number of features: 207
Time spent: 0.000s
-------------------------
Name: inq
Transformer type: None
Number of features: 10
Time spent: 0.000s
-------------------------
Name: member
Transformer type: None
Number of features: 32
Time spent: 0.000s
-------------------------
Name: trade
Transformer type: None
Number of features: 234
Time spent: 0.000s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 1
Time spent: 0.012s
-------------------------
Name: bnkr FE
Transformer type: 


INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:00:43.584146
INFO:zaml.artifact_engine.logger:Executing FinalDataArtifact <final_test_target>...
INFO:zaml.artifact_engine.logger:Finished <final_test_target>, total time spent: 0:00:00.000425
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 302
Time spent: 1.494s
-------------------------
Name: GeneralFeatureSelection
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.004s
-------------------------
Name: Drop Engineered Features
Transformer type: GeneralFeatureSelection
Number of features: 301
Time spent: 0.001s


INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.200591
INFO:zaml.artifact_engine.logger:Executing TestFeatureImportanceArtifact <feature_importance>...


INFO:zaml.artifact_engine.logger:Finished <feature_importance>, total time spent: 0:01:27.859257
INFO:zaml.artifact_engine.logger:Executing TestSubmodelFeatureImportanceArtifact <submodel_feature_importance>...



































INFO:zaml.artifact_engine.logger:Finished <submodel_feature_importance>, total time spent: 0:19:11.967986
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <test_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:00.140716
INFO:zaml.artifact_engine.logger:Executing FinalDataArtifact <final_test_scores>...
INFO:zaml.artifact_engine.logger:Finished <final_test_scores>, total time spent: 0:00:00.000556
INFO:zaml.artifact_engine.logger:Executing PerformanceArtifact <performance>...
INFO:zaml.artifact_engine

{'data_description': {'baseline_model': {'data_split': {'train': {'start_date': '2021-07-01',
     'end_date': '2023-04-21'},
    'test': {'start_date': '2023-04-21', 'end_date': '2023-11-01'}},
   'target': 'final_DQ60_m24',
   'data_sources': ['trade',
    'inq',
    'collec',
    'bnkr',
    'member',
    'app',
    'target',
    'config',
    'client_data'],
   'data_summary': {'train': {'num_of_rows': {'train_app': 44596}},
    'test': {'num_of_rows': {'test_app': 12293}}},
   'alias': 'client_model'},
  'submodel_0': {'data_split': {'train': {'start_date': '2016-04-01',
     'end_date': '2021-07-01'},
    'test': {'start_date': '2021-07-01', 'end_date': '2022-01-01'}},
   'target': 'final_DQ60_m24',
   'data_sources': ['app',
    'bnkr',
    'collec',
    'inq',
    'member',
    'target',
    'trade',
    'target',
    'config'],
   'data_summary': {'train': {'num_of_rows': {'train_app': 66744504}},
    'test': {'num_of_rows': {'test_app': 0}}}},
  'clients': {'clients': ['s3://

# Bonus

In [23]:
final_model_path = f'/d/shared/GYD_Modeling/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/final_model_{your_name}'


In [26]:
pipeline = load_state(os.path.join(final_model_path, 'pipeline.obj'))

In [27]:
for node in pipeline.graph.nodes:
    print(f"Node name: {node.name}, Node type: {type(node).__name__}")

Node name: app, Node type: InputNode
Node name: bnkr, Node type: InputNode
Node name: client_data, Node type: InputNode
Node name: collec, Node type: InputNode
Node name: config, Node type: InputNode
Node name: inq, Node type: InputNode
Node name: member, Node type: InputNode
Node name: trade, Node type: InputNode
Node name: app FE, Node type: TransformNode
Node name: bnkr FE, Node type: TransformNode
Node name: client_data FE, Node type: TransformNode
Node name: collec FE, Node type: TransformNode
Node name: config FE, Node type: TransformNode
Node name: inq FE, Node type: TransformNode
Node name: member FE, Node type: TransformNode
Node name: trade FE, Node type: TransformNode
Node name: Merge data, Node type: TransformNode
Node name: Bivariate Feature Engineering, Node type: TransformNode
Node name: Drop Features, Node type: TransformNode
Node name: OneHotEncoder, Node type: TransformNode
Node name: FillNA, Node type: TransformNode
Node name: GeneralFeatureSelection, Node type: Tran

# We can see that our model is a Linear Ensemble Model

In [29]:
pipeline.model

In [31]:
pipeline.model.__dict__.keys()

dict_keys(['_zaml_ver', '_other_params', 'explainer_type', 'models', 'weights', 'seed', 'model_name', 'calibration', 'predict_offset', 'normalize_predict_by_weight_sum', 'all_predictions'])

In [32]:
pipeline.model.models

[('XGBoostModel_0',
  <zaml.model.modeling.models.tree_models.XGBoostModel at 0x7fde01dc72b0>),
 ('XGBoostModel_1',
  <zaml.model.modeling.models.tree_models.XGBoostModel at 0x7fde01dc7460>)]